# AMEX Enterprise Credit Risk Platform
## Notebook 61 -- Real-Time Portfolio Monitoring: Financial-Impact Reporting & Packaging
### Phase 4 . Problem Statement 11: Real-Time Portfolio Monitoring

CRISP-DM stage: **Deployment / Business Reporting**. Sprint 1, Notebook 4 of 4 for this problem -- the final
notebook of Problem 11 and of Phase 4 (Operational Risk Management). Depends on Problem 1 Notebooks 01/05/08
(config, real champion metrics, real inherited EAD/LGD) and every notebook of this problem: 58 (policy), 59
(modeling), and 60 (validation & deployment).

**What this notebook does (real, computed on your machine when you run it):**
- Synthesizes real results from EVERY notebook of Problem 11 (58, 59, 60), not just this notebook's own
  financial calculations -- the elevated reporting standard this platform applies from Problem 7 onward
- Builds a financial model with a genuinely different SHAPE from every prior alerting problem's: because
  Problem 11's technique fires at the WHOLE-PORTFOLIO, calendar-month level (not per customer), responding
  to a real ALERT month carries an EVENT-LEVEL portfolio-risk-review cost (a credit-risk-committee
  convening, incurred once per real alert month) in addition to the familiar per-account cohort-review
  cost -- a cost stream with no analog in Problem 7's purely per-customer confusion-matrix pattern
- Computes real loss-prevention opportunity, net of both cost streams, from Notebook 60's real reproduced
  confusion matrix at the winning candidate and the real measured alert-month rate
- Computes ROI, investment, and payback period, and six SMART suggestions across organizational levels
- **Rebuilds the real alert feed a third independent time**: imports the EXACT
  `portfolio_alert_feed_service.py` Notebook 60 generated, seeds it with every real historical calendar
  month via its own real `POST /ingest-month` endpoint, and reads back its real `GET /alert-feed` response
  -- the identical JSON a live deployment's ops dashboard renders -- cross-checked against Notebook 59's
  persisted calendar-month count before use
- Reuses Notebook 59's four real charts and Notebook 60's two real charts directly (not regenerated), plus
  one new financial value-waterfall chart
- Assembles a Word report synthesizing maximum detail from every Problem 11 notebook, with a narrative
  "story" paragraph below every chart (standing user directive)
- Builds a colorful Excel workbook: Executive Summary KPI cards, live formulas, native AutoFilter tables,
  conditional formatting, an embedded chart, and a real Alert Feed sheet with ALERT months highlighted
- Builds an advanced, "global-standard" interactive HTML dashboard: multi-tab, with slicers, filters, a live
  financial calculator, full legends, interactive KPI cards -- and a dedicated **Alert Feed (Ops Dashboard)**
  tab that embeds the real `/alert-feed` snapshot from this run and additionally lets a viewer point it at a
  live-deployed instance of the service (optional `fetch()` against a real host + API key). This is what
  makes the dashboard double as the real ops dashboard + alert feed the master plan names as this problem's
  deliverable, not a mockup

**What this notebook does NOT do:** it selects no new candidate (that is Notebook 60's job) and computes no
new control-chart logic beyond the third independent reproduction in Section 8, which exists purely to
produce the real alert-feed JSON the dashboard displays.

**Honest edge-case handling:** every dollar figure is either read directly from a real, measured upstream
notebook artifact or is explicitly labeled ASSUMPTION with its rationale stated inline (matching the
platform's standing convention). If the winning candidate does not meet Notebook 58's KPI, every report
still honestly states NOT RECOMMENDED FOR PRODUCTION rather than hiding an unfavorable result. Verified end
to end (including a full Word-to-PDF render check, an Excel-workbook reload check, and a headless-browser
render of every dashboard tab with console-error inspection) before delivery.

Zero-fabrication: every real count and rate in this notebook comes directly from Notebook 58/59/60's
persisted JSON, or from this notebook's own third independent reproduction of the alert feed -- nothing is
re-estimated or guessed. Only the financial ASSUMPTION inputs (clearly labeled, with stated rationale) are
editable business judgment calls, exactly like every other financial-impact notebook in this platform.


In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOKS 05/08/58/59/60'S REAL OUTPUTS
#            (EVERY NOTEBOOK OF PROBLEM 11, PER THE ELEVATED REPORTING
#            STANDARD -- NOT JUST THIS NOTEBOOK'S OWN FINANCIAL CALCULATIONS)
# =============================================================================
import base64
import json
import os
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebooks 05/08/58/59/60's Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P11_ROOT = PROJECT_ROOT / "Phase4_Operational_Risk_Management" / "11_Problem11_Real_Time_Portfolio_Monitoring"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB58_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_58_summary.json"
NB59_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_59_summary.json"
NB60_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_60_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first."),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first."),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first (this notebook inherits its real EAD/LGD "
                         "assumptions rather than re-guessing them)."),
    (NB58_SUMMARY_PATH, "run 58_real_time_portfolio_monitoring_business_understanding.ipynb first."),
    (NB59_SUMMARY_PATH, "run 59_real_time_portfolio_monitoring_modeling.ipynb first."),
    (NB60_SUMMARY_PATH, "run 60_real_time_portfolio_monitoring_validation_deployment.ipynb first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected location.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB58_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB58_SUMMARY = json.load(f)
with open(NB59_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB59_SUMMARY = json.load(f)
with open(NB60_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB60_SUMMARY = json.load(f)

PORTFOLIO_MONITORING_POLICY_PATH = Path(NB58_SUMMARY["policy_path"])
with open(PORTFOLIO_MONITORING_POLICY_PATH, "r", encoding="utf-8") as f:
    PORTFOLIO_MONITORING_POLICY = json.load(f)

MODELING_RESULTS_PATH = Path(NB59_SUMMARY["modeling_results_path"])
with open(MODELING_RESULTS_PATH, "r", encoding="utf-8") as f:
    MODELING_RESULTS_ARTIFACT = json.load(f)

DEPLOYMENT_POLICY_PATH = Path(NB60_SUMMARY["deployment_policy_path"])
with open(DEPLOYMENT_POLICY_PATH, "r", encoding="utf-8") as f:
    DEPLOYMENT_POLICY = json.load(f)

SERVICE_PY_PATH = Path(NB60_SUMMARY["service_py_path"])
if not SERVICE_PY_PATH.exists():
    raise FileNotFoundError(f"{SERVICE_PY_PATH} not found.\nFix: re-run Notebook 60.")

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
if "portfolio_monitoring_reporting_packaging" in PILLAR_DIRS:
    P11_REPORTING_DIR = PILLAR_DIRS["portfolio_monitoring_reporting_packaging"]
else:
    P11_REPORTING_DIR = P11_ROOT / "financial_impact_reporting_packaging"
    print(f"NOTE: 'portfolio_monitoring_reporting_packaging' not in pillar_dirs -- using fallback: "
          f"{P11_REPORTING_DIR}")
P11_REPORTING_DIR.mkdir(parents=True, exist_ok=True)
# Same hardcoded (not pillar-driven) resolution Notebook 60 itself used for its own deployment dir --
# reused verbatim here so this notebook's chart paths deterministically match what Notebook 60 wrote.
P11_DEPLOYMENT_DIR = P11_ROOT / "deployment"

# --- Real values synthesized from EVERY notebook of Problem 11 (58, 59, 60),
#     per the elevated reporting standard -- not scoped to this notebook's
#     own financial calculations alone. ---
CONTROL_LIMIT_K_SIGMA = PORTFOLIO_MONITORING_POLICY["control_limit_k_sigma"]
MIN_TRAILING_MONTHS_FOR_BASELINE = PORTFOLIO_MONITORING_POLICY["min_trailing_months_for_baseline"]
MONITORED_BASE_COLUMNS = PORTFOLIO_MONITORING_POLICY["monitored_base_columns"]["columns"]
N_MONITORED_COLUMNS = len(MONITORED_BASE_COLUMNS)
PORTFOLIO_KPI_TARGETS = PORTFOLIO_MONITORING_POLICY["kpi_targets"]
P7_REFERENCE = PORTFOLIO_KPI_TARGETS["problem_7_reference"]

CANDIDATE_RESULTS = {int(k): v for k, v in MODELING_RESULTS_ARTIFACT["candidate_results"].items()}
ALERT_MONTHS_BY_CANDIDATE_STR = MODELING_RESULTS_ARTIFACT["alert_months_by_candidate"]
N_CALENDAR_MONTHS_COVERED = MODELING_RESULTS_ARTIFACT["n_calendar_months_covered"]
N_BASELINE_ELIGIBLE_MONTHS = MODELING_RESULTS_ARTIFACT["n_baseline_eligible_months"]
BASE_DEFAULT_RATE_HOLDOUT = MODELING_RESULTS_ARTIFACT["base_default_rate_holdout"]
SECONDARY_METRICS = MODELING_RESULTS_ARTIFACT["secondary_threshold_free_metrics"]

WINNING_CONSECUTIVE_BREACH_CANDIDATE = NB60_SUMMARY["winning_consecutive_breach_candidate"]
MEETS_KPI = NB60_SUMMARY["meets_kpi_target"]
RECOMMENDED_FOR_PRODUCTION = NB60_SUMMARY["recommended_for_production"]
WINNING_METRICS = NB60_SUMMARY["winning_candidate_metrics"]
LIFT_95CI = NB60_SUMMARY["lift_95ci"]
API_SELF_TEST_PASSED = NB60_SUMMARY["api_self_test_passed"]
API_LATENCY_SUMMARY = NB60_SUMMARY["api_latency_summary"]
RANDOM_SEED = NB60_SUMMARY["random_seed"]

N_ALERT_MONTHS_TOTAL = len(ALERT_MONTHS_BY_CANDIDATE_STR[str(WINNING_CONSECUTIVE_BREACH_CANDIDATE)])
ALERT_MONTH_RATE = (N_ALERT_MONTHS_TOTAL / N_BASELINE_ELIGIBLE_MONTHS) if N_BASELINE_ELIGIBLE_MONTHS else 0.0

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]

print("Real problem                                    : Real-Time Portfolio Monitoring (Phase 4, Problem 11)")
print(f"Winning CONSECUTIVE_BREACH_CANDIDATE (Notebook 60): {WINNING_CONSECUTIVE_BREACH_CANDIDATE}")
print(f"Meets KPI target / recommended for production    : {MEETS_KPI} / {RECOMMENDED_FOR_PRODUCTION}")
print(f"Real cohort default-rate lift (95% CI)            : "
      f"{(WINNING_METRICS['default_rate_lift'] or 0.0):.3f}x [{LIFT_95CI[0]}, {LIFT_95CI[1]}]")
print(f"Real calendar months covered / baseline-eligible  : {N_CALENDAR_MONTHS_COVERED} / {N_BASELINE_ELIGIBLE_MONTHS}")
print(f"Real ALERT months at the winning candidate        : {N_ALERT_MONTHS_TOTAL} "
      f"({ALERT_MONTH_RATE:.1%} of baseline-eligible months)")
print(f"API self-test passed (Notebook 60)                : {API_SELF_TEST_PASSED}")
print(f"EAD per account (Notebook 08, inherited)          : ${EAD_PER_ACCOUNT_USD:,}")
print(f"LGD assumption (Notebook 08, inherited)           : {LGD_ASSUMPTION:.0%}")
print(f"Problem 7 real reference (recommended / capture)  : "
      f"{P7_REFERENCE['recommended_for_production']} / {P7_REFERENCE['real_alert_capture_rate']:.1%}")
print(f"Reporting/packaging outputs will be written under : {P11_REPORTING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.chart import BarChart, Reference
except ImportError:
    missing.append("openpyxl")
try:
    import importlib.util
except ImportError:
    missing.append("importlib")
try:
    from fastapi.testclient import TestClient
except ImportError:
    missing.append("fastapi")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: FINANCIAL PLANNING ASSUMPTIONS (EXPLICIT, EDITABLE)
# =============================================================================
_section("SECTION 3: Financial Planning Assumptions (Explicit, Editable)")

# --- This dataset has no real intervention-outcome or project-cost data.
#     Every ASSUMPTION-labeled figure below is stated and editable -- nothing
#     here is fabricated as if it were measured. EAD/LGD are real inherited
#     values (read programmatically from Problem 1's Notebook 08). This
#     financial model has a genuinely different SHAPE from every prior
#     alerting problem's (5/6/7): Problem 11's technique fires at the WHOLE-
#     PORTFOLIO, CALENDAR-MONTH level, not per customer, so responding to it
#     carries an EVENT-LEVEL cost (a portfolio risk review, convened once per
#     real ALERT MONTH) in addition to the per-account cohort-review cost --
#     a cost stream with no analog in Problem 7's purely per-customer
#     confusion-matrix pattern. ---
FINANCIAL_ASSUMPTIONS = {
    "ead_per_account_usd": {"value": EAD_PER_ACCOUNT_USD,
                             "source": "Notebook 08 (inherited, real value read programmatically)"},
    "lgd_assumption": {"value": LGD_ASSUMPTION,
                        "source": "Notebook 08 (inherited, real value read programmatically)"},
    "cohort_intervention_success_rate": {
        "value": 0.12,
        "source": "ASSUMPTION -- illustrative efficacy of intensified monitoring/collections outreach applied "
                   "to the WHOLE COHORT of customers whose latest statement falls in a real ALERT month; set "
                   "below Problem 7's 15% per-customer intervention rate because this response is triggered "
                   "by a coarser, whole-portfolio signal applied to an entire monthly cohort rather than a "
                   "customer-specific behavioral deviation -- edit to your institution's own outcome data.",
    },
    "cohort_review_cost_usd_per_account": {
        "value": 12,
        "source": "ASSUMPTION -- illustrative lightweight per-account triage cost for a customer in a "
                   "flagged monthly cohort who does NOT go on to default; set below Problem 7's $15 since "
                   "these accounts are triaged as part of a batch cohort review process (a monthly list), "
                   "not individually investigated -- edit to your institution's actual cost.",
    },
    "portfolio_risk_review_cost_usd_per_alert_month": {
        "value": 2500,
        "source": "ASSUMPTION -- illustrative one-time credit-risk-committee convening plus root-cause "
                   "investigation cost incurred EACH TIME the control chart declares a real ALERT month -- "
                   "a genuinely new cost stream with no Problem 5/6/7 analog, reflecting this technique's "
                   "whole-portfolio, calendar-month event trigger rather than a per-customer signal; edit "
                   "to your institution's actual review-committee cost.",
    },
    "implementation_cost_usd": {
        "value": 38_000,
        "source": "ASSUMPTION -- illustrative one-time build/validate/deploy cost for the real-time "
                   "portfolio-monitoring streaming-aggregation service and its ops dashboard + alert feed; "
                   "set above Problem 7's $30,000 (a stateless per-request scorer) because this service "
                   "must maintain server-side running history (the alert feed) and an ops dashboard, and "
                   "below Problem 6's $60,000 model-training pipeline since this technique trains no model "
                   "at all -- edit to your institution's actual project cost.",
    },
    "annual_application_cycles": {
        "value": 12,
        "source": "ASSUMPTION -- monthly re-aggregation cadence, matching this technique's own real "
                   "calendar-month grain (see Notebook 58); edit to your institution's actual cadence.",
    },
}
COHORT_INTERVENTION_SUCCESS_RATE = FINANCIAL_ASSUMPTIONS["cohort_intervention_success_rate"]["value"]
COHORT_REVIEW_COST_USD_PER_ACCOUNT = FINANCIAL_ASSUMPTIONS["cohort_review_cost_usd_per_account"]["value"]
PORTFOLIO_RISK_REVIEW_COST_USD_PER_ALERT_MONTH = (
    FINANCIAL_ASSUMPTIONS["portfolio_risk_review_cost_usd_per_alert_month"]["value"]
)
IMPLEMENTATION_COST_USD = FINANCIAL_ASSUMPTIONS["implementation_cost_usd"]["value"]
ANNUAL_APPLICATION_CYCLES = FINANCIAL_ASSUMPTIONS["annual_application_cycles"]["value"]

assumptions_path = P11_REPORTING_DIR / "financial_assumptions.json"
with open(assumptions_path, "w", encoding="utf-8") as f:
    json.dump(FINANCIAL_ASSUMPTIONS, f, indent=2)
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    print(f"  {_k}: {_v['value']}  ({_v['source'][:90]}...)")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL ALERT VALUE -- POPULATION FLAGGED (EXACT, FROM NOTEBOOK 60'S
#            REPRODUCED CONFUSION MATRIX AT THE WINNING CANDIDATE) AND REAL
#            ALERT-MONTH FREQUENCY
# =============================================================================
_section("SECTION 4: Real Alert Value -- Population Flagged and Alert-Month Frequency")

_cm_winning = WINNING_METRICS["confusion_matrix"]
N_HOLDOUT_EVAL = sum(_cm_winning.values())
N_HOLDOUT_DEFAULTERS = _cm_winning["tp"] + _cm_winning["fn"]
TRUE_POSITIVES_FLAGGED = _cm_winning["tp"]
FALSE_POSITIVES_FLAGGED = _cm_winning["fp"]
FLAGGED_TOTAL = TRUE_POSITIVES_FLAGGED + FALSE_POSITIVES_FLAGGED
COHORT_CAPTURE_RATE = TRUE_POSITIVES_FLAGGED / N_HOLDOUT_DEFAULTERS if N_HOLDOUT_DEFAULTERS else 0.0

print(f"Winning CONSECUTIVE_BREACH_CANDIDATE (Notebook 60)      : {WINNING_CONSECUTIVE_BREACH_CANDIDATE}")
print(f"Real holdout evaluation population (cohort-eligible)     : {N_HOLDOUT_EVAL:,}")
print(f"Real holdout defaulters (tp + fn, exact)                 : {N_HOLDOUT_DEFAULTERS:,}")
print(f"Real true positives flagged (tp, exact)                  : {TRUE_POSITIVES_FLAGGED:,}")
print(f"Real false positives flagged (fp, exact)                 : {FALSE_POSITIVES_FLAGGED:,}")
print(f"Total cohort customers flagged for review                : {FLAGGED_TOTAL:,}")
print(f"Real cohort defaulter capture rate at this candidate      : {COHORT_CAPTURE_RATE:.1%}")
print(f"Real cohort default-rate lift (Notebook 60, 95% CI)       : "
      f"{(WINNING_METRICS['default_rate_lift'] or 0.0):.3f}x [{LIFT_95CI[0]}, {LIFT_95CI[1]}]")
print(f"Real ALERT months (of {N_BASELINE_ELIGIBLE_MONTHS} baseline-eligible)               : "
      f"{N_ALERT_MONTHS_TOTAL} ({ALERT_MONTH_RATE:.1%})")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: LOSS-PREVENTION OPPORTUNITY, NET OF COHORT-REVIEW COST AND THE
#            EVENT-LEVEL PORTFOLIO RISK REVIEW COST
# =============================================================================
_section("SECTION 5: Loss-Prevention Opportunity, Net of Cohort-Review and Portfolio Risk Review Costs")

PREVENTABLE_DEFAULTS = round(TRUE_POSITIVES_FLAGGED * COHORT_INTERVENTION_SUCCESS_RATE)
GROSS_LOSS_PREVENTED_USD = PREVENTABLE_DEFAULTS * EAD_PER_ACCOUNT_USD * LGD_ASSUMPTION
COHORT_REVIEW_COST_USD = FALSE_POSITIVES_FLAGGED * COHORT_REVIEW_COST_USD_PER_ACCOUNT
# --- Expected cost per cycle: not every monthly cycle is a real ALERT month,
#     so the event-level portfolio risk review cost is weighted by the real,
#     measured ALERT_MONTH_RATE rather than charged every cycle. ---
PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD = ALERT_MONTH_RATE * PORTFOLIO_RISK_REVIEW_COST_USD_PER_ALERT_MONTH
NET_BENEFIT_PER_CYCLE_USD = (
    GROSS_LOSS_PREVENTED_USD - COHORT_REVIEW_COST_USD - PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD
)

print(f"True positives flagged (real, exact)                     : {TRUE_POSITIVES_FLAGGED:,}")
print(f"ASSUMPTION cohort-intervention success rate               : {COHORT_INTERVENTION_SUCCESS_RATE:.0%}")
print(f"Estimated preventable defaults                            : {PREVENTABLE_DEFAULTS:,}")
print(f"Gross loss prevented (this holdout sample, per cycle)     : ${GROSS_LOSS_PREVENTED_USD:,.0f}")
print(f"False positives flagged (real, exact)                     : {FALSE_POSITIVES_FLAGGED:,}")
print(f"ASSUMPTION cost per cohort-review account                 : ${COHORT_REVIEW_COST_USD_PER_ACCOUNT}")
print(f"Total cohort-review cost (per cycle)                      : ${COHORT_REVIEW_COST_USD:,.0f}")
print(f"Real ALERT-month rate (measured)                          : {ALERT_MONTH_RATE:.1%}")
print(f"ASSUMPTION portfolio risk review cost per real alert month: "
      f"${PORTFOLIO_RISK_REVIEW_COST_USD_PER_ALERT_MONTH:,}")
print(f"Expected portfolio risk review cost (per cycle)           : ${PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD:,.0f}")
print(f"\nNet benefit per cycle (gross loss prevented - cohort review - portfolio risk review): "
      f"${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: ROI, INVESTMENT & PAYBACK PERIOD
# =============================================================================
_section("SECTION 6: ROI, Investment & Payback Period")

ANNUAL_BENEFIT_USD = NET_BENEFIT_PER_CYCLE_USD * ANNUAL_APPLICATION_CYCLES
ROI_PCT = ((ANNUAL_BENEFIT_USD - IMPLEMENTATION_COST_USD) / IMPLEMENTATION_COST_USD) * 100 if IMPLEMENTATION_COST_USD else None
PAYBACK_MONTHS = (IMPLEMENTATION_COST_USD / (ANNUAL_BENEFIT_USD / 12)) if ANNUAL_BENEFIT_USD > 0 else None
if ANNUAL_BENEFIT_USD > 0:
    ROI_DISPLAY = f"{ROI_PCT:,.0f}%"
    PAYBACK_DISPLAY = f"{PAYBACK_MONTHS:.1f} months"
    ROI_PCT_JSON = round(ROI_PCT, 1)
    PAYBACK_MONTHS_JSON = round(PAYBACK_MONTHS, 2)
else:
    ROI_DISPLAY = "N/A -- no measurable net benefit under current assumptions"
    PAYBACK_DISPLAY = "N/A -- no measurable net benefit under current assumptions"
    ROI_PCT_JSON = None
    PAYBACK_MONTHS_JSON = None

print(f"Amount invested (ASSUMPTION)         : ${IMPLEMENTATION_COST_USD:,.0f}")
print(f"Annual net benefit ({ANNUAL_APPLICATION_CYCLES}x/year cadence): ${ANNUAL_BENEFIT_USD:,.0f}")
print(f"Estimated Year-1 ROI                  : {ROI_DISPLAY}")
print(f"Estimated payback period              : {PAYBACK_DISPLAY}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SMART SUGGESTIONS -- BOTTOM TO TOP MANAGEMENT
# =============================================================================
_section("SECTION 7: SMART Suggestions -- Bottom to Top Management")

SMART_SUGGESTIONS = [
    {"org_level": "Portfolio Monitoring Ops / Frontline",
     "suggestion": f"Watch the live ops dashboard's Alert Feed tab every real calendar month as new "
                   f"statement batches land -- when a month clears CONSECUTIVE_BREACH_CANDIDATE="
                   f"{WINNING_CONSECUTIVE_BREACH_CANDIDATE} consecutive breaching months, pull the "
                   f"{FLAGGED_TOTAL:,}-account cohort whose latest statement falls in that month for "
                   f"intensified monitoring/collections outreach -- this is a whole-COHORT action, not a "
                   f"per-customer alert queue like Problem 7's."},
    {"org_level": "Portfolio Risk Team Lead",
     "suggestion": f"Convene the portfolio risk review each time an alert month fires ({N_ALERT_MONTHS_TOTAL:,} "
                   f"real alert months of {N_BASELINE_ELIGIBLE_MONTHS:,} baseline-eligible months on this run, "
                   f"{ALERT_MONTH_RATE:.1%}) -- track the {PREVENTABLE_DEFAULTS:,}-account intervention goal "
                   f"against the {FALSE_POSITIVES_FLAGGED:,} real false positives (cohort-review cost "
                   f"exposure) as paired monthly KPIs, the same true-positive/false-positive tension every "
                   f"alerting technique in this platform trades, at this technique's own portfolio-month "
                   f"grain."},
    {"org_level": "Risk / Credit Analyst",
     "suggestion": f"Monitor the real cohort default-rate lift ({(WINNING_METRICS['default_rate_lift'] or 0.0):.2f}x, "
                   f"95% CI [{LIFT_95CI[0]}, {LIFT_95CI[1]}]) every cycle -- a coarse, month-level signal by "
                   f"construction (see Notebook 58's honesty note), so the CI width matters more here than "
                   f"for a per-customer technique; also watch the split-half score PSI Notebook 60 computed "
                   f"for drift in the underlying monitored columns themselves."},
    {"org_level": "Model Risk / Compliance (SR 11-7)",
     "suggestion": f"File Notebook 60's bootstrap lift CI, statistical validation table, and API self-test "
                   f"result ({API_SELF_TEST_PASSED}) with the technique's annual governance packet; note this "
                   f"is an unsupervised, rule-based statistical-process-control technique operating on "
                   f"WHOLE-PORTFOLIO aggregates (not a trained classifier, and not a per-customer alert -- "
                   f"that is Problem 7), so governance review should assess CONTROL_LIMIT_K_SIGMA and "
                   f"MIN_TRAILING_MONTHS_FOR_BASELINE stability, not model retraining cadence. Currently "
                   f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production."},
    {"org_level": "Finance / Provisioning Team",
     "suggestion": "Use real ALERT months as a trigger for an EARLY, whole-cohort reserve-timing review -- "
                   "a portfolio-level complement to Problem 7's per-customer alert-triggered reviews; "
                   "coordinate with Problem 3's ECL work and Problem 4's tier-differentiated LGD for the $ "
                   "reserve amount per flagged cohort account."},
    {"org_level": "CFO / Executive Leadership",
     "suggestion": f"Approve the ${IMPLEMENTATION_COST_USD:,.0f} investment given an estimated "
                   f"{PAYBACK_DISPLAY} payback and {ROI_DISPLAY} Year-1 ROI (net of estimated cohort-review "
                   f"and portfolio-risk-review costs) from monthly-equivalent alert triage; the running ops "
                   f"dashboard + alert feed this notebook packages is the master plan's own literal "
                   f"deliverable for this problem."},
]
smart_df = pd.DataFrame(SMART_SUGGESTIONS)
smart_path = P11_REPORTING_DIR / "p11_smart_suggestions.csv"
smart_df.to_csv(smart_path, index=False)
for _row in SMART_SUGGESTIONS:
    print(f"[{_row['org_level']}]\n  {_row['suggestion']}\n")
print(f"\u2705 Saved -> {smart_path.name}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: REBUILD THE REAL ALERT FEED VIA THE DEPLOYED SERVICE -- THIS IS
#            WHAT MAKES THE HTML DASHBOARD BELOW DOUBLE AS THE REAL OPS
#            DASHBOARD + ALERT FEED THE MASTER PLAN NAMES AS THIS PROBLEM'S
#            DELIVERABLE, NOT A MOCKUP
# =============================================================================
_section("SECTION 8: Rebuild the Real Alert Feed via the Deployed Service")

print(
    "Rather than hand-assemble alert-feed data for the dashboard below, this section imports the EXACT "
    "portfolio_alert_feed_service.py Notebook 60 generated and saved, seeds it with every real historical "
    "calendar month (via its own real POST /ingest-month endpoint, in real chronological order), and reads "
    "back its real GET /alert-feed response -- the identical JSON a live deployment's ops dashboard would "
    "render. This is a third independent reproduction of the monthly portfolio store (after Notebooks 59 "
    "and 60), so it is cross-checked against Notebook 59's persisted calendar-month count before use."
)


def build_monthly_portfolio_store(csv_path: Path, monitored_cols: list) -> "pl.DataFrame":
    """Identical to Notebooks 59/60's function -- see Notebook 59 for the full docstring. Reused verbatim."""
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in monitored_cols:
        schema_overrides[c] = pl.Float32
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in monitored_cols
    ]
    agg_exprs = [pl.len().alias("n_statements"), pl.col("customer_ID").n_unique().alias("n_unique_customers")]
    agg_exprs += [pl.col(c).mean().alias(f"{c}_portfolio_mean") for c in monitored_cols]
    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .select(["customer_ID", "S_2"] + monitored_cols)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d").dt.truncate("1mo").alias("_month"))
        .with_columns(_inf_clean_exprs)
        .group_by("_month")
        .agg(agg_exprs)
        .sort("_month")
    )
    return lf.collect(engine="streaming")


_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TEST_DATA_PATH = RAW_TRAIN_DATA_PATH.parent / "test_data.csv"
_HAS_TEST_DATA = RAW_TEST_DATA_PATH.exists() and RAW_TEST_DATA_PATH.stat().st_size > 1_000_000

_train_monthly = build_monthly_portfolio_store(RAW_TRAIN_DATA_PATH, MONITORED_BASE_COLUMNS)
_train_monthly = _train_monthly.with_columns(pl.lit("train").alias("_source"))
if _HAS_TEST_DATA:
    _test_monthly = build_monthly_portfolio_store(RAW_TEST_DATA_PATH, MONITORED_BASE_COLUMNS)
    _test_monthly = _test_monthly.with_columns(pl.lit("test").alias("_source"))
    _train_months_set = set(_train_monthly["_month"].to_list())
    _test_monthly = _test_monthly.filter(~pl.col("_month").is_in(_train_months_set))
    MONTHLY_PORTFOLIO_STORE = pl.concat([_train_monthly, _test_monthly], how="vertical").sort("_month")
else:
    MONTHLY_PORTFOLIO_STORE = _train_monthly

_reproduced_n_months = MONTHLY_PORTFOLIO_STORE.height
_reported_n_months = MODELING_RESULTS_ARTIFACT["n_calendar_months_covered"]
print(f"Reproduced calendar-month count: {_reproduced_n_months}  (Notebook 59 reported {_reported_n_months})")
if _reproduced_n_months != _reported_n_months:
    raise RuntimeError(
        f"Notebook 61's reproduced calendar-month count ({_reproduced_n_months}) does NOT match Notebook "
        f"59's persisted count ({_reported_n_months}) -- investigate before proceeding."
    )

_sorted_store = MONTHLY_PORTFOLIO_STORE.sort("_month")
_months_list = _sorted_store["_month"].to_list()
_n_months = len(_months_list)
_col_series = {c: _sorted_store[f"{c}_portfolio_mean"].to_numpy().astype(np.float64) for c in MONITORED_BASE_COLUMNS}

os.environ["AMEX_P11_POLICY_PATH"] = str(DEPLOYMENT_POLICY_PATH)
_DASHBOARD_API_KEY = "notebook-61-dashboard-seed-key"
os.environ["API_KEY"] = _DASHBOARD_API_KEY
_dash_headers = {"X-API-Key": _DASHBOARD_API_KEY}

_spec = importlib.util.spec_from_file_location("amex_portfolio_alert_feed_service_nb61", str(SERVICE_PY_PATH))
_service_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_service_module)
dash_client = TestClient(_service_module.app)

for _i in range(_n_months):
    _payload = {
        "month": str(_months_list[_i]),
        "n_statements": int(_sorted_store["n_statements"][_i]),
        "n_unique_customers": int(_sorted_store["n_unique_customers"][_i]),
        "column_means": {
            c: (None if np.isnan(_col_series[c][_i]) else float(_col_series[c][_i])) for c in MONITORED_BASE_COLUMNS
        },
    }
    _resp = dash_client.post("/ingest-month", json=_payload, headers=_dash_headers)
    if _resp.status_code != 200:
        raise RuntimeError(f"/ingest-month failed seeding the real ops dashboard: {_resp.status_code} {_resp.text}")
print(f"Seeded the live deployed service with all {_n_months} real calendar months via POST /ingest-month.")

_feed_resp = dash_client.get("/alert-feed", headers=_dash_headers)
if _feed_resp.status_code != 200:
    raise RuntimeError(f"GET /alert-feed failed: {_feed_resp.status_code} {_feed_resp.text}")
ALERT_FEED = _feed_resp.json()
if ALERT_FEED["n_months"] != _n_months:
    raise RuntimeError(
        f"Live /alert-feed reports {ALERT_FEED['n_months']} months, expected {_n_months} -- investigate "
        "before proceeding."
    )
_live_n_alert_months = sum(1 for _m in ALERT_FEED["months"] if _m["alert"])
print(f"Real live /alert-feed response: {ALERT_FEED['n_months']} months, {_live_n_alert_months} in a real "
      f"ALERT state -- this exact JSON structure is what a live deployment's ops dashboard renders, and is "
      f"embedded directly into Section 12's HTML dashboard below.")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: CONSOLIDATE CHARTS FROM NOTEBOOKS 59/60 + NEW FINANCIAL CHART
# =============================================================================
_section("SECTION 9: Consolidate Charts From Notebooks 59/60 + New Financial Chart")

# --- Per the elevated reporting standard, this problem's Word/HTML reports
#     reuse the REAL chart PNGs Notebooks 59 and 60 already rendered (not
#     regenerated) -- the same "reuse real charts directly" pattern this
#     platform established from Problem 6 onward. Only ONE new chart is
#     generated here: the financial value-waterfall chart, which has no
#     earlier-notebook equivalent. ---
NB59_TREND_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["monthly_kpi_trend"])
NB59_ROC_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["roc_curve"])
NB59_PR_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["pr_curve"])
NB59_LIFT_CHART_PATH = Path(MODELING_RESULTS_ARTIFACT["chart_paths"]["lift_by_candidate"])
NB60_BOOTSTRAP_CHART_PATH = P11_DEPLOYMENT_DIR / "charts" / "notebook_60_bootstrap_lift_distribution.png"
NB60_CALIBRATION_CHART_PATH = P11_DEPLOYMENT_DIR / "charts" / "notebook_60_calibration_by_score_bin.png"

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "good": "#16a34a", "bad": "#dc2626",
       "gold": "#C9A227", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7.5, 4.5), dpi=150)
_waterfall_labels = ["Gross Loss\nPrevented", "Cohort Review\nCost", "Portfolio Risk\nReview Cost", "Net Benefit"]
_waterfall_vals = [GROSS_LOSS_PREVENTED_USD, -COHORT_REVIEW_COST_USD, -PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD,
                    NET_BENEFIT_PER_CYCLE_USD]
_waterfall_colors = [VIZ["good"], VIZ["bad"], VIZ["bad"], VIZ["gold"]]
_bars = ax1.bar(_waterfall_labels, _waterfall_vals, color=_waterfall_colors)
for _b, _v in zip(_bars, _waterfall_vals):
    ax1.text(_b.get_x() + _b.get_width() / 2, _v, f"${_v:,.0f}",
              ha="center", va="bottom" if _v >= 0 else "top", fontsize=9)
ax1.axhline(0, color=VIZ["ink"], linewidth=0.8)
ax1.set_ylabel("USD per cycle (real population, ASSUMPTION $ inputs)")
ax1.set_title(f"Problem 11: Net Benefit Components @ Candidate={WINNING_CONSECUTIVE_BREACH_CANDIDATE}")
fig1.tight_layout()
chart_financial_path = P11_REPORTING_DIR / "net_benefit_waterfall_chart.png"
fig1.savefig(chart_financial_path, dpi=150, facecolor=VIZ["surface"])
plt.close(fig1)

_reused_charts_present = {
    "trend": NB59_TREND_CHART_PATH.exists(), "roc": NB59_ROC_CHART_PATH.exists(),
    "pr": NB59_PR_CHART_PATH.exists(), "lift": NB59_LIFT_CHART_PATH.exists(),
    "bootstrap": NB60_BOOTSTRAP_CHART_PATH.exists(), "calibration": NB60_CALIBRATION_CHART_PATH.exists(),
}
print(f"\u2705 Saved -> {chart_financial_path.name} (new)")
for _name, _path in [("Monthly trend (Notebook 59)", NB59_TREND_CHART_PATH),
                      ("ROC (Notebook 59)", NB59_ROC_CHART_PATH), ("PR (Notebook 59)", NB59_PR_CHART_PATH),
                      ("Lift-by-candidate (Notebook 59)", NB59_LIFT_CHART_PATH),
                      ("Bootstrap lift (Notebook 60)", NB60_BOOTSTRAP_CHART_PATH),
                      ("Calibration (Notebook 60)", NB60_CALIBRATION_CHART_PATH)]:
    print(f"  Reused -> {_name}: {'found' if _path.exists() else 'MISSING'} ({_path})")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: WORD REPORT (ELEVATED) -- SYNTHESIZES MAXIMUM DETAIL FROM EVERY
#             NOTEBOOK OF PROBLEM 11 (58, 59, 60), NOT JUST THIS NOTEBOOK'S
#             OWN FINANCIAL CALCULATIONS -- EVERY CHART FOLLOWED BY A STORY
#             PARAGRAPH (STANDING USER DIRECTIVE)
# =============================================================================
_section("SECTION 10: Word Report (Elevated) -- Real_Time_Portfolio_Monitoring_Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


def _add_chart_with_story(doc, chart_path: Path, caption: str, story: str):
    if not chart_path.exists():
        doc.add_paragraph(f"[Chart not found: {chart_path.name} -- re-run the notebook that produces it.]")
        return
    doc.add_picture(str(chart_path), width=Inches(6.0))
    _cap = doc.add_paragraph()
    _cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    _run = _cap.add_run(caption)
    _run.bold = True
    _run.font.size = Pt(10)
    _story_p = doc.add_paragraph(story)
    _story_p.paragraph_format.space_after = Pt(14)


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 4, Problem 11: Real-Time Portfolio Monitoring -- Comprehensive Financial Impact Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
doc.add_paragraph(
    "This report synthesizes real results from EVERY notebook of Problem 11 -- Notebook 58 (Business "
    "Understanding & Policy), Notebook 59 (Modeling), Notebook 60 (Validation & Deployment), and this "
    "notebook's own financial-impact calculations -- per the platform's elevated reporting standard. Every "
    "figure is real and measured except values explicitly labeled ASSUMPTION, which are editable business "
    "inputs."
)

_add_heading(doc, "1. Executive Summary", level=1)
doc.add_paragraph(
    f"The whole-portfolio, calendar-time control-chart technique validated in Notebooks 58-60 flags a real "
    f"calendar month whose portfolio-wide monitored KPIs deviate from the portfolio's OWN recent trailing "
    f"baseline for {WINNING_CONSECUTIVE_BREACH_CANDIDATE} or more consecutive months -- the same statistical-"
    f"process-control idea Problem 7 established, elevated from the per-customer axis to the whole-portfolio, "
    f"calendar-month axis. This technique is currently "
    f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production. At the winning "
    f"candidate, {N_ALERT_MONTHS_TOTAL} of {N_BASELINE_ELIGIBLE_MONTHS} real baseline-eligible months "
    f"({ALERT_MONTH_RATE:.1%}) were in a real ALERT state, and holdout customers whose latest statement falls "
    f"in an alert month show a real default-rate lift of {(WINNING_METRICS['default_rate_lift'] or 0.0):.2f}x "
    f"(95% CI [{LIFT_95CI[0]}, {LIFT_95CI[1]}]) against the >= {PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x "
    f"KPI target. At an ASSUMPTION {COHORT_INTERVENTION_SUCCESS_RATE:.0%} cohort-intervention success rate, "
    f"net of ASSUMPTION per-account cohort-review cost and event-level portfolio-risk-review cost, this is "
    f"estimated to net ${NET_BENEFIT_PER_CYCLE_USD:,.0f} of benefit per monthly-equivalent cycle, for an "
    f"estimated {PAYBACK_DISPLAY} payback on a ${IMPLEMENTATION_COST_USD:,.0f} implementation investment."
)

_add_heading(doc, "2. Business Understanding & Policy (Notebook 58)", level=1)
doc.add_paragraph(
    "Problem 11 asks a genuinely different question from Problem 7's per-customer question: 'does THIS "
    "MONTH's whole-book average of a headline KPI look different from the PORTFOLIO's OWN recent trailing "
    "baseline?' A real per-column, per-month EXPANDING trailing baseline is computed from all real strictly-"
    "prior calendar months; a month is 'breaching' when at least one monitored column's z-score clears the "
    "control limit; a month is a real ALERT month when it ends a consecutive run of breaching months at "
    "least as long as the winning candidate."
)
_add_kv_table(doc, {
    "control_limit_k_sigma_assumption": CONTROL_LIMIT_K_SIGMA,
    "min_trailing_months_for_baseline_assumption": MIN_TRAILING_MONTHS_FOR_BASELINE,
    "monitored_base_columns": ", ".join(MONITORED_BASE_COLUMNS),
    "monitored_column_count": N_MONITORED_COLUMNS,
    "monitored_column_source": "Reused from Problem 1's real SHAP global importance ranking (Notebook 06), "
                                "with an honest fallback to Notebook 05's native feature importance",
    "real_calendar_months_covered": N_CALENDAR_MONTHS_COVERED,
    "real_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "primary_kpi": f">= {PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x cohort default-rate lift "
                   "(ASSUMPTION target)",
    "problem_7_reference_recommended": P7_REFERENCE["recommended_for_production"],
    "problem_7_reference_alert_capture_rate": f"{P7_REFERENCE['real_alert_capture_rate']:.1%}",
})
_add_chart_with_story(
    doc, NB59_TREND_CHART_PATH,
    "Figure 1. Real Monthly Portfolio Trend With Control-Limit Breaches (Notebook 59)",
    "This chart traces every monitored column's real portfolio-wide monthly mean across the full real "
    "calendar-month history, with real breaching months marked -- the first whole-portfolio, calendar-time "
    "aggregation in this platform, as opposed to every prior notebook's per-customer or per-split view."
)

_add_heading(doc, "3. Modeling -- Candidate Sweep Results (Notebook 59)", level=1)
doc.add_paragraph(
    "Notebook 59 computed the real trailing-baseline control chart across all real calendar months and swept "
    "every CONSECUTIVE_BREACH_CANDIDATE, reporting the full classification metrics suite at each -- treating "
    "'cohort in a flagged month' as the binary prediction, per the platform's standing metrics-suite "
    "directive."
)
_candidate_rows = []
for _c in sorted(CANDIDATE_RESULTS.keys()):
    _m = CANDIDATE_RESULTS[_c]
    _candidate_rows.append({
        "candidate": _c, "n_alerted": _m["n_alerted"], "pct_alerted": f"{_m['pct_alerted']:.2f}%",
        "default_rate_lift": f"{(_m['default_rate_lift'] or 0.0):.3f}x",
        "meets_kpi": _m["meets_kpi_target"], "precision": round(_m["precision"], 4),
        "recall": round(_m["recall"], 4), "f1": round(_m["f1"], 4), "mcc": round(_m["mcc"], 4),
    })
_candidate_df = pd.DataFrame(_candidate_rows)
_add_table_from_df(doc, _candidate_df)
doc.add_paragraph(
    f"Secondary, non-gating threshold-free metrics of the continuous normalized monthly breach-count score: "
    f"ROC-AUC {SECONDARY_METRICS['roc_auc']}, PR-AUC {SECONDARY_METRICS['pr_auc']}. A lower AUC than "
    "Problems 1/5/6/7's trained classifiers or per-customer techniques is the honestly expected outcome for "
    "a coarse, month-level aggregate signal, not a failure of this notebook."
)
_add_chart_with_story(
    doc, NB59_ROC_CHART_PATH, "Figure 2. ROC Curve -- Continuous Monthly Breach-Count Score (Notebook 59)",
    "Traces true-positive rate against false-positive rate as the normalized monthly breach-count score is "
    "swept as a continuous ranking signal, independent of any specific candidate threshold. A coarse, "
    "month-level aggregate signal is not designed to rank-order individual customers, so an AUC near the "
    "random-chance diagonal is expected and reported honestly as a secondary comparability metric."
)
_add_chart_with_story(
    doc, NB59_PR_CHART_PATH, "Figure 3. Precision-Recall Curve -- Continuous Monthly Breach-Count Score (Notebook 59)",
    f"The more informative counterpart to the ROC curve on this imbalanced dataset (base default rate "
    f"{BASE_DEFAULT_RATE_HOLDOUT:.1%}): shows how precision trades off against recall as the score threshold "
    "moves, compared against the no-skill base-rate baseline."
)
_add_chart_with_story(
    doc, NB59_LIFT_CHART_PATH, "Figure 4. Real Cohort Default-Rate Lift by Candidate (Notebook 59)",
    f"This is the technique's PRIMARY KPI across every real candidate threshold swept. The dashed line marks "
    f"the >= {PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x KPI target. Notebook 60 selects the "
    "winning candidate directly from this real sweep -- the smallest candidate clearing the bar, or, when "
    "none do, the candidate with the real highest lift, flagged NOT RECOMMENDED FOR PRODUCTION."
)

_add_heading(doc, "4. Validation & Deployment (Notebook 60)", level=1)
doc.add_paragraph(
    f"Notebook 60 deterministically rebuilt Notebook 59's entire computation (zero randomness) and reproduced "
    f"its numbers exactly as an integrity check, then selected the winning candidate: "
    f"CONSECUTIVE_BREACH_CANDIDATE={WINNING_CONSECUTIVE_BREACH_CANDIDATE}, "
    + ("the smallest candidate clearing the KPI." if MEETS_KPI else
       "the candidate with the real highest default-rate lift, since none of the tested candidates cleared "
       "the KPI on this run -- flagged NOT RECOMMENDED FOR PRODUCTION throughout.")
)
_add_kv_table(doc, {
    "winning_consecutive_breach_candidate": WINNING_CONSECUTIVE_BREACH_CANDIDATE,
    "n_alerted": WINNING_METRICS["n_alerted"], "pct_alerted": f"{WINNING_METRICS['pct_alerted']:.2f}%",
    "default_rate_lift_point_estimate": f"{(WINNING_METRICS['default_rate_lift'] or 0.0):.3f}x",
    "default_rate_lift_95pct_ci": f"[{LIFT_95CI[0]}, {LIFT_95CI[1]}]",
    "meets_kpi_target": MEETS_KPI, "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "api_self_test_passed": API_SELF_TEST_PASSED,
    "api_p50_p99_latency_ms": f"{API_LATENCY_SUMMARY['p50_ms']} / {API_LATENCY_SUMMARY['p99_ms']}",
})
_add_chart_with_story(
    doc, NB60_BOOTSTRAP_CHART_PATH,
    f"Figure 5. Bootstrap Distribution -- Cohort Default-Rate Lift @ Candidate={WINNING_CONSECUTIVE_BREACH_CANDIDATE} "
    "(Notebook 60)",
    "Notebook 60 resampled the holdout evaluation population 2,000 times (with replacement) and recomputed "
    "the lift each time -- this histogram is the resulting distribution. The width of the resulting 95% CI "
    "directly reflects statistical uncertainty around the point estimate."
)
_add_chart_with_story(
    doc, NB60_CALIBRATION_CHART_PATH, "Figure 6. Score-Rank Calibration -- Real Holdout Cohort Bins (Notebook 60)",
    "Bins holdout cohorts by their normalized monthly breach-count score and plots each bin's real observed "
    "default rate, testing whether a higher score tracks a higher real default rate -- the ranking property "
    "an alerting system needs, not strict predicted-probability calibration."
)

_add_heading(doc, "5. Real Alert Value: Population Flagged and Alert-Month Frequency", level=1)
doc.add_paragraph(
    "Every count in this section comes directly from Notebook 60's reproduced confusion matrix at the "
    "winning candidate on the real holdout population, and from the real alert-month list Notebook 59 "
    "computed -- exact, not derived or estimated."
)
_add_kv_table(doc, {
    "winning_consecutive_breach_candidate": WINNING_CONSECUTIVE_BREACH_CANDIDATE,
    "real_holdout_evaluation_population": f"{N_HOLDOUT_EVAL:,}",
    "real_holdout_defaulters": f"{N_HOLDOUT_DEFAULTERS:,}",
    "true_positives_flagged": f"{TRUE_POSITIVES_FLAGGED:,}",
    "false_positives_flagged": f"{FALSE_POSITIVES_FLAGGED:,}",
    "real_cohort_defaulter_capture_rate": f"{COHORT_CAPTURE_RATE:.1%}",
    "real_alert_months": f"{N_ALERT_MONTHS_TOTAL} of {N_BASELINE_ELIGIBLE_MONTHS} baseline-eligible "
                          f"({ALERT_MONTH_RATE:.1%})",
})

_add_heading(doc, "6. Loss-Prevention Opportunity, Net of Costs", level=1)
_add_kv_table(doc, {
    "true_positives_flagged": TRUE_POSITIVES_FLAGGED,
    "cohort_intervention_success_rate_assumption": f"{COHORT_INTERVENTION_SUCCESS_RATE:.0%}",
    "preventable_defaults": PREVENTABLE_DEFAULTS,
    "gross_loss_prevented_per_cycle_usd": f"${GROSS_LOSS_PREVENTED_USD:,.0f}",
    "false_positives_flagged": FALSE_POSITIVES_FLAGGED,
    "cohort_review_cost_per_account_assumption": f"${COHORT_REVIEW_COST_USD_PER_ACCOUNT}",
    "total_cohort_review_cost_usd": f"${COHORT_REVIEW_COST_USD:,.0f}",
    "real_alert_month_rate": f"{ALERT_MONTH_RATE:.1%}",
    "portfolio_risk_review_cost_per_alert_month_assumption": f"${PORTFOLIO_RISK_REVIEW_COST_USD_PER_ALERT_MONTH:,}",
    "expected_portfolio_risk_review_cost_per_cycle_usd": f"${PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD:,.0f}",
    "net_benefit_per_cycle_usd": f"${NET_BENEFIT_PER_CYCLE_USD:,.0f}",
})
_add_chart_with_story(
    doc, chart_financial_path, f"Figure 7. Net Benefit Components @ Candidate={WINNING_CONSECUTIVE_BREACH_CANDIDATE} "
    "(Notebook 61)",
    "This waterfall shows exactly how the net benefit per cycle in Section 6 above is composed: gross loss "
    "prevented (green) less the per-account cohort-review cost and the expected event-level portfolio-risk-"
    "review cost (both red), landing at the net benefit (gold) -- the two red cost bars are this problem's "
    "own genuinely different financial-model shape versus Problem 7's purely per-customer confusion-matrix "
    "pattern."
)

_add_heading(doc, "7. ROI, Investment & Payback", level=1)
_add_kv_table(doc, {"amount_invested_usd": f"${IMPLEMENTATION_COST_USD:,.0f}",
                     "annual_net_benefit_usd": f"${ANNUAL_BENEFIT_USD:,.0f}",
                     "roi_year_1_pct": ROI_DISPLAY, "payback_period_months": PAYBACK_DISPLAY})

_add_heading(doc, "8. Real Ops Dashboard + Alert Feed (Notebook 61)", level=1)
doc.add_paragraph(
    f"Section 8 of this notebook seeded the exact portfolio_alert_feed_service.py Notebook 60 generated with "
    f"all {_n_months} real calendar months and read back its real GET /alert-feed response -- "
    f"{_live_n_alert_months} of {_n_months} months are in a real ALERT state. That exact JSON is embedded in "
    "the interactive HTML dashboard packaged alongside this report, which doubles as the real ops dashboard "
    "+ alert feed the master plan names as this problem's deliverable."
)

_add_heading(doc, "9. SMART Suggestions by Organizational Level", level=1)
_add_table_from_df(doc, smart_df)

_add_heading(doc, "10. Assumptions & Sources", level=1)
_assump_df = pd.DataFrame([{"assumption": k, "value": v["value"], "source": v["source"]}
                            for k, v in FINANCIAL_ASSUMPTIONS.items()])
_add_table_from_df(doc, _assump_df)

_add_heading(doc, "11. Final Recommendation & Deployment Status", level=1)
doc.add_paragraph(
    f"Overall deployment status: "
    f"{'RECOMMENDED FOR PRODUCTION' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED FOR PRODUCTION'}. "
    + ("This technique clears its KPI, statistical validation, and API self-test bars -- deploy per Notebook "
       "60's deployment readiness checklist." if RECOMMENDED_FOR_PRODUCTION else
       "This is the best-performing candidate tested on this run and is packaged here for completeness "
       "(policy artifact, real-time alert-feed service, full validation, ops dashboard) so the platform's "
       "tooling exists end to end -- but it should not be deployed to production until a future run either "
       "finds a candidate that clears the KPI or the KPI target itself is revisited with the business "
       "stakeholder.")
)
doc.add_paragraph(
    "This closes out Phase 4 (Operational Risk Management) and Problem 11 -- the final problem of this "
    "platform's master plan."
)

report_path = P11_REPORTING_DIR / "Real_Time_Portfolio_Monitoring_Financial_Impact_Report.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path.name}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: EXCEL WORKBOOK -- COLORFUL, TABLE + AUTOFILTER + CONDITIONAL
#             FORMATTING + CHART
# =============================================================================
_section("SECTION 11: Excel Workbook -- Colorful, Table + AutoFilter + Conditional Formatting + Chart")

INK = "0B1F3A"
ACCENT = "C41E3A"
GOLD = "C9A227"
LIGHT = "F2F4F8"
WHITE = "FFFFFF"
GOOD = "63BE7B"
USD_FMT = '$#,##0;($#,##0);-'

_assump_rows = {k: 2 + i for i, k in enumerate(FINANCIAL_ASSUMPTIONS.keys())}

wb = openpyxl.Workbook()

# --- Sheet: Assumptions ---
ws_assump = wb.active
ws_assump.title = "Assumptions"
ws_assump.append(["Assumption", "Value", "Source / Rationale"])
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    ws_assump.append([_k.replace("_", " ").title(), _v["value"], _v["source"]])
for _r in range(2, ws_assump.max_row + 1):
    ws_assump[f"B{_r}"].fill = PatternFill("solid", fgColor="FFFF00")
    ws_assump[f"B{_r}"].font = Font(name="Calibri", color="0000FF")
    ws_assump[f"C{_r}"].alignment = Alignment(wrap_text=True, vertical="top")
ws_assump[f"B{_assump_rows['lgd_assumption']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['cohort_intervention_success_rate']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['ead_per_account_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['cohort_review_cost_usd_per_account']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['portfolio_risk_review_cost_usd_per_alert_month']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['implementation_cost_usd']}"].number_format = USD_FMT
ws_assump.column_dimensions["A"].width = 42
ws_assump.column_dimensions["B"].width = 14
ws_assump.column_dimensions["C"].width = 90
_tbl_assump = Table(displayName="Assumptions", ref=f"A1:C{ws_assump.max_row}")
_tbl_assump.tableStyleInfo = TableStyleInfo(name="TableStyleMedium4", showRowStripes=True)
ws_assump.add_table(_tbl_assump)

_ead_ref = f"Assumptions!$B${_assump_rows['ead_per_account_usd']}"
_lgd_ref = f"Assumptions!$B${_assump_rows['lgd_assumption']}"
_success_rate_ref = f"Assumptions!$B${_assump_rows['cohort_intervention_success_rate']}"
_review_cost_ref = f"Assumptions!$B${_assump_rows['cohort_review_cost_usd_per_account']}"
_portfolio_review_cost_ref = f"Assumptions!$B${_assump_rows['portfolio_risk_review_cost_usd_per_alert_month']}"
_cost_ref = f"Assumptions!$B${_assump_rows['implementation_cost_usd']}"
_cycles_ref = f"Assumptions!$B${_assump_rows['annual_application_cycles']}"

# --- Sheet: Alert Impact ---
ws_impact = wb.create_sheet("Alert Impact")
ws_impact.append(["Metric", "Value"])
_impact_rows_static = [
    ("Winning CONSECUTIVE_BREACH_CANDIDATE (Notebook 60)", WINNING_CONSECUTIVE_BREACH_CANDIDATE),
    ("Real Holdout Evaluation Population", N_HOLDOUT_EVAL),
    ("Real Holdout Defaulters (exact)", N_HOLDOUT_DEFAULTERS),
    ("True Positives Flagged (exact)", TRUE_POSITIVES_FLAGGED),
    ("False Positives Flagged (exact)", FALSE_POSITIVES_FLAGGED),
    ("Real Cohort Defaulter Capture Rate", COHORT_CAPTURE_RATE),
    ("Real Alert-Month Rate", ALERT_MONTH_RATE),
]
for _label, _val in _impact_rows_static:
    ws_impact.append([_label, _val])
_preventable_row = ws_impact.max_row + 1
ws_impact.append(["Preventable Defaults", f"=ROUND(B5*{_success_rate_ref},0)"])
_gross_loss_row = ws_impact.max_row + 1
ws_impact.append(["Gross Loss Prevented / Cycle (USD)", f"=B{_preventable_row}*{_ead_ref}*{_lgd_ref}"])
_review_cost_row = ws_impact.max_row + 1
ws_impact.append(["Cohort Review Cost / Cycle (USD)", f"=B6*{_review_cost_ref}"])
_portfolio_review_row = ws_impact.max_row + 1
ws_impact.append(["Portfolio Risk Review Cost / Cycle (USD, expected)", f"=B8*{_portfolio_review_cost_ref}"])
_net_benefit_row = ws_impact.max_row + 1
ws_impact.append(["Net Benefit / Cycle (USD)",
                   f"=B{_gross_loss_row}-B{_review_cost_row}-B{_portfolio_review_row}"])
_annual_benefit_row = ws_impact.max_row + 1
ws_impact.append(["Annual Net Benefit (USD)", f"=B{_net_benefit_row}*{_cycles_ref}"])
ws_impact["B7"].number_format = "0.0%"
ws_impact["B8"].number_format = "0.0%"
ws_impact[f"B{_gross_loss_row}"].number_format = USD_FMT
ws_impact[f"B{_review_cost_row}"].number_format = USD_FMT
ws_impact[f"B{_portfolio_review_row}"].number_format = USD_FMT
ws_impact[f"B{_net_benefit_row}"].number_format = USD_FMT
ws_impact[f"B{_annual_benefit_row}"].number_format = USD_FMT
ws_impact.column_dimensions["A"].width = 46
ws_impact.column_dimensions["B"].width = 20
_tbl_impact = Table(displayName="AlertImpact", ref=f"A1:B{ws_impact.max_row}")
_tbl_impact.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws_impact.add_table(_tbl_impact)

_chart = BarChart()
_chart.title = "Flagged Population: True Positives vs. False Positives"
_chart.y_axis.title = "Customers"
_data = Reference(ws_impact, min_col=2, min_row=1, max_row=6)
_cats = Reference(ws_impact, min_col=1, min_row=2, max_row=6)
_chart.add_data(_data, titles_from_data=True)
_chart.set_categories(_cats)
_chart.width, _chart.height = 20, 10
ws_impact.add_chart(_chart, "D2")

# --- Sheet: Candidate Sweep (Notebook 59's real results, native AutoFilter) ---
ws_sweep = wb.create_sheet("Candidate Sweep (NB59)")
ws_sweep.append(["Candidate", "N Alerted", "% Alerted", "Default Rate Lift", "Meets KPI",
                  "Precision", "Recall", "F1", "MCC"])
for _c in sorted(CANDIDATE_RESULTS.keys()):
    _m = CANDIDATE_RESULTS[_c]
    ws_sweep.append([_c, _m["n_alerted"], _m["pct_alerted"] / 100.0, _m["default_rate_lift"] or 0.0,
                      "Yes" if _m["meets_kpi_target"] else "No", _m["precision"], _m["recall"],
                      _m["f1"], _m["mcc"]])
_last_row_sweep = ws_sweep.max_row
for _r in range(2, _last_row_sweep + 1):
    ws_sweep[f"C{_r}"].number_format = "0.00%"
    ws_sweep[f"D{_r}"].number_format = '0.00"x"'
_tbl_sweep = Table(displayName="CandidateSweep", ref=f"A1:I{_last_row_sweep}")
_tbl_sweep.tableStyleInfo = TableStyleInfo(name="TableStyleMedium6", showRowStripes=True)
ws_sweep.add_table(_tbl_sweep)
for _col, _w in zip("ABCDEFGHI", [12, 11, 11, 16, 11, 11, 10, 10, 10]):
    ws_sweep.column_dimensions[_col].width = _w

# --- Sheet: Alert Feed (real, live, from the deployed service -- Section 8) ---
ws_feed = wb.create_sheet("Alert Feed (Live)")
ws_feed.append(["Month", "N Statements", "N Unique Customers", "Baseline Eligible", "Breach Count",
                 "Consecutive Run Length", "Alert"])
for _m in ALERT_FEED["months"]:
    ws_feed.append([_m["month"], _m["n_statements"], _m["n_unique_customers"], _m["baseline_eligible"],
                     _m["breach_count"], _m["consecutive_breach_run_length"], "Yes" if _m["alert"] else "No"])
_last_row_feed = ws_feed.max_row
_tbl_feed = Table(displayName="AlertFeed", ref=f"A1:G{_last_row_feed}")
_tbl_feed.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_feed.add_table(_tbl_feed)
for _col, _w in zip("ABCDEFG", [12, 13, 17, 15, 12, 18, 8]):
    ws_feed.column_dimensions[_col].width = _w
_red_fill = PatternFill("solid", fgColor="FFC7CE")
for _r in range(2, _last_row_feed + 1):
    if ws_feed[f"G{_r}"].value == "Yes":
        for _col in "ABCDEFG":
            ws_feed[f"{_col}{_r}"].fill = _red_fill

# --- Sheet: SMART Suggestions (real Excel Table -> native AutoFilter dropdowns) ---
ws_smart = wb.create_sheet("SMART Suggestions")
ws_smart.append(["Org Level", "Suggestion"])
for _row_data in SMART_SUGGESTIONS:
    ws_smart.append([_row_data["org_level"], _row_data["suggestion"]])
_last_row_smart = ws_smart.max_row
_tbl_smart = Table(displayName="SmartSuggestions", ref=f"A1:B{_last_row_smart}")
_tbl_smart.tableStyleInfo = TableStyleInfo(name="TableStyleMedium7", showRowStripes=True)
ws_smart.add_table(_tbl_smart)
ws_smart.column_dimensions["A"].width = 40
ws_smart.column_dimensions["B"].width = 100
for _r in range(2, _last_row_smart + 1):
    ws_smart[f"B{_r}"].alignment = Alignment(wrap_text=True, vertical="top")

# --- Sheet: Executive Summary (KPI cards), inserted first, populated last ---
ws_exec = wb.create_sheet("Executive Summary", 0)
wb.active = 0
ws_exec.sheet_view.showGridLines = False
ws_exec["B2"] = "AMEX RiskIQ -- Problem 11: Real-Time Portfolio Monitoring"
ws_exec["B2"].font = Font(name="Calibri", size=16, bold=True, color=WHITE)
ws_exec["B2"].fill = PatternFill("solid", fgColor=INK)
ws_exec.merge_cells("B2:F2")
ws_exec["B3"] = "Comprehensive Financial Impact Summary"
ws_exec["B3"].font = Font(name="Calibri", size=11, italic=True, color=INK)
ws_exec.merge_cells("B3:F3")

_kpi_rows = [
    ("Winning Candidate", f"CONSECUTIVE_BREACH_CANDIDATE={WINNING_CONSECUTIVE_BREACH_CANDIDATE}  "
                           "(see Candidate Sweep sheet)", False, LIGHT),
    ("Defaulters Captured (Exact)", "='Alert Impact'!B4", True, LIGHT),
    ("Real Alert Months", "='Alert Impact'!B7", True, LIGHT),
    ("Net Benefit / Cycle", f"='Alert Impact'!B{_net_benefit_row}", True, GOLD),
    ("Amount Invested", f"=\"$\"&TEXT({_cost_ref},\"#,##0\")", True, LIGHT),
    ("Estimated Year-1 ROI", f"{ROI_DISPLAY}  (reported, see Section 7)", False, ACCENT),
    ("Estimated Payback", f"{PAYBACK_DISPLAY}  (reported, see Section 7)", False, ACCENT),
    ("Deployment Status", f"{'RECOMMENDED' if RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} for production",
     False, GOOD if RECOMMENDED_FOR_PRODUCTION else ACCENT),
]
_row = 5
for _label, _value, _is_formula, _fill in _kpi_rows:
    ws_exec.cell(row=_row, column=2, value=_label).font = Font(name="Calibri", size=11, color=INK)
    _cell = ws_exec.cell(row=_row, column=4, value=_value)
    _cell.font = Font(name="Calibri", size=13, bold=True, color=(WHITE if _fill in (GOLD, ACCENT, GOOD) else INK))
    _cell.fill = PatternFill("solid", fgColor=_fill)
    _cell.alignment = Alignment(horizontal="center", wrap_text=not _is_formula)
    if _is_formula and _label in ("Defaulters Captured (Exact)", "Real Alert Months"):
        _cell.number_format = "#,##0"
    elif _is_formula and "Benefit" in _label:
        _cell.number_format = USD_FMT
    ws_exec.merge_cells(start_row=_row, start_column=4, end_row=_row, end_column=5)
    _row += 1
ws_exec["B14"] = "Rows recalculate live from the Assumptions and Alert Impact sheets."
ws_exec["B14"].font = Font(name="Calibri", size=9, italic=True, color="8A93A6")
ws_exec.merge_cells("B14:F14")
for _col, _w in zip("BCDEF", [32, 3, 22, 22, 3]):
    ws_exec.column_dimensions[_col].width = _w

for _ws in (ws_impact, ws_sweep, ws_feed, ws_smart, ws_assump):
    for _cell in _ws[1]:
        _cell.font = Font(name="Calibri", bold=True, color=WHITE)
        _cell.fill = PatternFill("solid", fgColor=INK)

workbook_path = P11_REPORTING_DIR / "AMEX_Problem11_Financial_Impact_Workbook.xlsx"
wb.save(str(workbook_path))
print(f"\u2705 Saved -> {workbook_path.name}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: INTERACTIVE HTML DASHBOARD (ELEVATED) -- GLOBAL-STANDARD,
#             MULTI-TAB, WITH SLICERS, FILTERS, A LIVE FINANCIAL CALCULATOR,
#             FULL LEGENDS, AND A REAL ALERT-FEED TAB (DOUBLES AS THE OPS
#             DASHBOARD + ALERT FEED THE MASTER PLAN NAMES AS THIS PROBLEM'S
#             DELIVERABLE)
# =============================================================================
_section("SECTION 12: Interactive HTML Dashboard (Elevated) -- Doubles as the Real Ops Dashboard + Alert Feed")


def _b64_image(path: Path) -> str:
    if not path.exists():
        return ""
    return base64.b64encode(path.read_bytes()).decode("ascii")


_trend_b64 = _b64_image(NB59_TREND_CHART_PATH)
_roc_b64 = _b64_image(NB59_ROC_CHART_PATH)
_pr_b64 = _b64_image(NB59_PR_CHART_PATH)
_lift_b64 = _b64_image(NB59_LIFT_CHART_PATH)
_bootstrap_b64 = _b64_image(NB60_BOOTSTRAP_CHART_PATH)
_calibration_b64 = _b64_image(NB60_CALIBRATION_CHART_PATH)
_financial_b64 = _b64_image(chart_financial_path)

_candidate_records = []
for _c in sorted(CANDIDATE_RESULTS.keys()):
    _m = CANDIDATE_RESULTS[_c]
    _candidate_records.append({
        "candidate": _c, "n_alerted": _m["n_alerted"], "pct_alerted": round(_m["pct_alerted"], 2),
        "lift": round(_m["default_rate_lift"] or 0.0, 3), "meets_kpi": bool(_m["meets_kpi_target"]),
        "accuracy": round(_m["accuracy"], 4), "precision": round(_m["precision"], 4),
        "recall": round(_m["recall"], 4), "f1": round(_m["f1"], 4), "specificity": round(_m["specificity"], 4),
        "mcc": round(_m["mcc"], 4), "is_winner": _c == WINNING_CONSECUTIVE_BREACH_CANDIDATE,
    })
_org_levels = sorted({r["org_level"] for r in SMART_SUGGESTIONS})
_feed_records = [
    {"month": _m["month"], "n_statements": _m["n_statements"], "n_unique_customers": _m["n_unique_customers"],
     "baseline_eligible": _m["baseline_eligible"], "breach_count": _m["breach_count"],
     "run_length": _m["consecutive_breach_run_length"], "alert": _m["alert"]}
    for _m in ALERT_FEED["months"]
]

_calc_constants = {
    "tp": TRUE_POSITIVES_FLAGGED, "fp": FALSE_POSITIVES_FLAGGED, "ead": EAD_PER_ACCOUNT_USD,
    "lgd": LGD_ASSUMPTION, "alert_month_rate": ALERT_MONTH_RATE,
    "default_success_rate": COHORT_INTERVENTION_SUCCESS_RATE, "default_review_cost": COHORT_REVIEW_COST_USD_PER_ACCOUNT,
    "default_portfolio_review_cost": PORTFOLIO_RISK_REVIEW_COST_USD_PER_ALERT_MONTH,
    "default_cycles": ANNUAL_APPLICATION_CYCLES, "default_impl_cost": IMPLEMENTATION_COST_USD,
}

_policy_kv = [
    ("CONTROL_LIMIT_K_SIGMA (ASSUMPTION)", CONTROL_LIMIT_K_SIGMA),
    ("MIN_TRAILING_MONTHS_FOR_BASELINE (ASSUMPTION)", MIN_TRAILING_MONTHS_FOR_BASELINE),
    ("Monitored Base Columns", ", ".join(MONITORED_BASE_COLUMNS)),
    ("Real Calendar Months Covered", N_CALENDAR_MONTHS_COVERED),
    ("Real Baseline-Eligible Months", N_BASELINE_ELIGIBLE_MONTHS),
    ("Primary KPI", f">= {PORTFOLIO_KPI_TARGETS['min_cohort_default_rate_lift']}x cohort default-rate lift"),
    ("Problem 7 Reference (Recommended / Capture Rate)",
     f"{P7_REFERENCE['recommended_for_production']} / {P7_REFERENCE['real_alert_capture_rate']:.1%}"),
]
_validation_kv = [
    ("Winning Candidate", WINNING_CONSECUTIVE_BREACH_CANDIDATE),
    ("N Alerted / % Alerted", f"{WINNING_METRICS['n_alerted']:,} / {WINNING_METRICS['pct_alerted']:.2f}%"),
    ("Default-Rate Lift (Point / 95% CI)",
     f"{(WINNING_METRICS['default_rate_lift'] or 0.0):.3f}x / [{LIFT_95CI[0]}, {LIFT_95CI[1]}]"),
    ("API Self-Test Passed", API_SELF_TEST_PASSED),
    ("API Latency p50 / p99 (ms)", f"{API_LATENCY_SUMMARY['p50_ms']} / {API_LATENCY_SUMMARY['p99_ms']}"),
    ("Real Alert Months / Baseline-Eligible", f"{N_ALERT_MONTHS_TOTAL} / {N_BASELINE_ELIGIBLE_MONTHS}"),
]

_html = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Problem 11 -- Real-Time Portfolio Monitoring Ops Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>
<style>
  :root { --ink:#0B1F3A; --accent:#C41E3A; --gold:#C9A227; --muted:#8A93A6; --bg:#F2F4F8; --card:#FFFFFF; --good:#16a34a; --bad:#dc2626; }
  * { box-sizing: border-box; }
  body { font-family: Calibri, Arial, sans-serif; background: var(--bg); color: var(--ink); margin: 0; padding: 24px; }
  h1 { font-size: 22px; margin-bottom: 4px; }
  h2 { font-size: 17px; margin-top: 0; }
  .sub { color: var(--muted); margin-bottom: 20px; }
  .kpi-row { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 20px; }
  .kpi { background: var(--card); border-radius: 10px; padding: 14px 18px; box-shadow: 0 1px 3px rgba(0,0,0,.12); min-width: 170px; flex: 1; transition: transform .15s; }
  .kpi:hover { transform: translateY(-2px); box-shadow: 0 4px 10px rgba(0,0,0,.16); }
  .kpi .label { font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: .03em; }
  .kpi .value { font-size: 20px; font-weight: 700; margin-top: 4px; }
  .kpi .sub2 { font-size: 11px; color: var(--muted); margin-top: 2px; }
  .tabs { display: flex; gap: 4px; margin-bottom: 16px; border-bottom: 2px solid #E4E7EE; flex-wrap: wrap; }
  .tab-btn { background: none; border: none; padding: 10px 16px; font-size: 13px; font-weight: 600; color: var(--muted); cursor: pointer; border-bottom: 3px solid transparent; }
  .tab-btn.active { color: var(--ink); border-bottom-color: var(--accent); }
  .tab-panel { display: none; }
  .tab-panel.active { display: block; }
  .panel { background: var(--card); border-radius: 10px; padding: 18px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E4E7EE; }
  th { background: var(--ink); color: #fff; position: sticky; top: 0; }
  tr.winner-row { background: #FFF7E6; font-weight: 700; }
  tr.alert-row { background: #FDECEC; }
  select, input[type=range], input[type=text] { padding: 6px 10px; border-radius: 6px; border: 1px solid var(--muted); font-size: 13px; }
  input[type=checkbox] { transform: scale(1.2); margin-right: 6px; }
  canvas { max-height: 380px; }
  .badge { display: inline-block; padding: 3px 10px; border-radius: 12px; font-size: 12px; font-weight: 700; }
  .legend-row { display: flex; gap: 18px; flex-wrap: wrap; font-size: 12px; color: var(--muted); margin-top: 8px; }
  .legend-swatch { display: inline-block; width: 10px; height: 10px; border-radius: 2px; margin-right: 5px; vertical-align: middle; }
  .filter-row { display: flex; gap: 16px; flex-wrap: wrap; align-items: center; margin-bottom: 14px; }
  .metric-btn { padding: 6px 12px; border-radius: 6px; border: 1px solid var(--muted); background: #fff; font-size: 12px; cursor: pointer; }
  .metric-btn.active { background: var(--ink); color: #fff; border-color: var(--ink); }
  .calc-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
  .calc-slider-row { margin-bottom: 18px; }
  .calc-slider-row label { display: block; font-size: 12px; color: var(--muted); margin-bottom: 4px; }
  .calc-slider-row .val { font-weight: 700; color: var(--ink); }
  .calc-out { background: var(--bg); border-radius: 8px; padding: 14px; }
  .calc-out .row { display: flex; justify-content: space-between; padding: 6px 0; border-bottom: 1px dashed #D6DAE4; font-size: 13px; }
  .calc-out .row.total { font-weight: 700; font-size: 15px; color: var(--accent); border-bottom: none; }
  .chart-story { font-size: 12.5px; color: #3a4560; margin-top: 10px; line-height: 1.5; }
  img.report-chart { width: 100%; max-width: 720px; display: block; margin: 0 auto; border-radius: 6px; }
  .live-row { display: flex; gap: 10px; align-items: center; margin-bottom: 14px; flex-wrap: wrap; }
  .live-status { font-size: 12px; color: var(--muted); }
  @media (max-width: 900px) { .calc-grid { grid-template-columns: 1fr; } }
</style>
</head>
<body>
<h1>AMEX RiskIQ -- Problem 11: Real-Time Portfolio Monitoring</h1>
<div class="sub">Whole-Portfolio Streaming Aggregation + Threshold Alerting -- real Notebook 58-60 results synthesized here, ASSUMPTION values clearly marked and adjustable in the Financial Calculator tab</div>

<div class="kpi-row">
  <div class="kpi"><div class="label">Winning Candidate</div><div class="value">__WINNING_CANDIDATE__</div><div class="sub2">of __N_CANDIDATES__ tested</div></div>
  <div class="kpi"><div class="label">Real Alert Months</div><div class="value">__N_ALERT_MONTHS__</div><div class="sub2">of __N_ELIGIBLE_MONTHS__ eligible (__ALERT_MONTH_RATE__)</div></div>
  <div class="kpi"><div class="label">Cohort Default-Rate Lift</div><div class="value">__LIFT_POINT__</div><div class="sub2">95% CI __LIFT_CI__</div></div>
  <div class="kpi"><div class="label">Net Benefit / Cycle</div><div class="value">__NET_BENEFIT__</div><div class="sub2">ASSUMPTION-driven, adjustable</div></div>
  <div class="kpi"><div class="label">Est. Year-1 ROI</div><div class="value">__ROI__</div><div class="sub2">Est. payback __PAYBACK__</div></div>
  <div class="kpi"><div class="label">Deployment Status</div><div class="value"><span class="badge" style="background:__STATUS_COLOR__;color:#fff;">__STATUS__</span></div></div>
</div>

<div class="tabs">
  <button class="tab-btn active" data-tab="overview">Overview</button>
  <button class="tab-btn" data-tab="alertfeed">Alert Feed (Ops Dashboard)</button>
  <button class="tab-btn" data-tab="policy">Policy (NB58)</button>
  <button class="tab-btn" data-tab="modeling">Modeling (NB59)</button>
  <button class="tab-btn" data-tab="validation">Validation (NB60)</button>
  <button class="tab-btn" data-tab="calculator">Financial Calculator</button>
  <button class="tab-btn" data-tab="smart">SMART Suggestions</button>
</div>

<div id="tab-overview" class="tab-panel active">
  <div class="panel">
    <h2>Candidate Sweep -- Real Cohort Default-Rate Lift &amp; Metrics (Notebook 59)</h2>
    <div class="filter-row">
      <label><input type="checkbox" id="kpiOnlyFilter"> Show only candidates meeting the KPI (slicer)</label>
      <span id="metricButtons"></span>
    </div>
    <canvas id="candidateChart"></canvas>
    <div class="legend-row" id="candidateLegend"></div>
    <table id="candidateTable">
      <thead><tr><th>Candidate</th><th>N Alerted</th><th>% Alerted</th><th>Lift</th><th>Meets KPI</th>
      <th>Precision</th><th>Recall</th><th>F1</th><th>MCC</th></tr></thead>
      <tbody></tbody>
    </table>
  </div>
  <div class="panel">
    <h2>Net Benefit Components (Notebook 61)</h2>
    <img class="report-chart" src="data:image/png;base64,__FINANCIAL_B64__" alt="Net benefit waterfall chart">
    <p class="chart-story">Adjust the sliders in the Financial Calculator tab to see how each component -- and the
    resulting net benefit -- responds to different ASSUMPTION inputs.</p>
  </div>
</div>

<div id="tab-alertfeed" class="tab-panel">
  <div class="panel">
    <h2>Real Alert Feed -- Live From the Deployed Service</h2>
    <p class="chart-story">This table is the EXACT <code>GET /alert-feed</code> response from
    <code>portfolio_alert_feed_service.py</code> (Notebook 60's generated deployment artifact), seeded with every
    real calendar month this run found. This tab doubles as the real ops dashboard the master plan names as this
    problem's deliverable -- it is not a mockup.</p>
    <div class="live-row">
      <label for="liveApiBase"><b>Point at a live deployment (optional):</b></label>
      <input type="text" id="liveApiBase" placeholder="https://your-deployment-host:8011" style="min-width:280px;">
      <input type="text" id="liveApiKey" placeholder="X-API-Key" style="min-width:160px;">
      <button class="metric-btn" id="liveFetchBtn">Fetch Live Feed</button>
      <span class="live-status" id="liveStatus">Showing the embedded snapshot from this real notebook run.</span>
    </div>
    <div class="filter-row">
      <label><input type="checkbox" id="alertOnlyFilter"> Show only real ALERT months (slicer)</label>
    </div>
    <table id="feedTable">
      <thead><tr><th>Month</th><th>N Statements</th><th>N Unique Customers</th><th>Baseline Eligible</th>
      <th>Breach Count</th><th>Consecutive Run Length</th><th>Alert</th></tr></thead>
      <tbody></tbody>
    </table>
  </div>
</div>

<div id="tab-policy" class="tab-panel">
  <div class="panel">
    <h2>Business Understanding &amp; Policy (Notebook 58)</h2>
    <p class="chart-story">Problem 11 flags a real calendar month whose portfolio-wide monitored KPIs deviate from
    the portfolio's OWN recent trailing baseline for enough consecutive months at once -- the same statistical-
    process-control idea Problem 7 established, elevated from the per-customer axis to the whole-portfolio,
    calendar-month axis.</p>
    <table id="policyTable"><tbody></tbody></table>
  </div>
  <div class="panel">
    <h2>Real Monthly Portfolio Trend With Control-Limit Breaches</h2>
    <img class="report-chart" src="data:image/png;base64,__TREND_B64__" alt="Monthly trend chart">
    <p class="chart-story">Every monitored column's real portfolio-wide monthly mean across the full real
    calendar-month history, with real breaching months marked -- the first whole-portfolio, calendar-time
    aggregation in this platform.</p>
  </div>
</div>

<div id="tab-modeling" class="tab-panel">
  <div class="panel">
    <h2>ROC Curve -- Continuous Monthly Breach-Count Score</h2>
    <img class="report-chart" src="data:image/png;base64,__ROC_B64__" alt="ROC curve">
    <p class="chart-story">Traces true-positive rate against false-positive rate as the normalized score is swept
    as a continuous ranking signal. A lower AUC than a trained classifier is the honestly expected outcome for this
    coarse, month-level aggregate technique.</p>
  </div>
  <div class="panel">
    <h2>Precision-Recall Curve -- Continuous Monthly Breach-Count Score</h2>
    <img class="report-chart" src="data:image/png;base64,__PR_B64__" alt="Precision-Recall curve">
    <p class="chart-story">The more informative counterpart to the ROC curve on this imbalanced dataset: shows how
    precision trades off against recall as the threshold moves, compared against the no-skill base-rate baseline.</p>
  </div>
  <div class="panel">
    <h2>Real Cohort Default-Rate Lift by Candidate</h2>
    <img class="report-chart" src="data:image/png;base64,__LIFT_B64__" alt="Lift by candidate chart">
    <p class="chart-story">The technique's PRIMARY KPI across every candidate threshold. Notebook 60 selects the
    winning candidate directly from this real sweep -- see the interactive version of this same data in the
    Overview tab above, filterable by KPI status and switchable by metric.</p>
  </div>
</div>

<div id="tab-validation" class="tab-panel">
  <div class="panel">
    <h2>Statistical Validation Summary (Notebook 60)</h2>
    <table id="validationTable"><tbody></tbody></table>
  </div>
  <div class="panel">
    <h2>Bootstrap Distribution -- Cohort Default-Rate Lift at the Winning Candidate</h2>
    <img class="report-chart" src="data:image/png;base64,__BOOTSTRAP_B64__" alt="Bootstrap lift distribution">
    <p class="chart-story">2,000 resamples of the holdout evaluation population, lift recomputed each time. The
    resulting 95% CI width reflects real statistical uncertainty.</p>
  </div>
  <div class="panel">
    <h2>Score-Rank Calibration -- Real Holdout Cohort Bins</h2>
    <img class="report-chart" src="data:image/png;base64,__CALIBRATION_B64__" alt="Calibration by score bin">
    <p class="chart-story">Tests whether a higher score tracks a higher real default rate, monotonically -- the
    ranking property an alerting system needs, not strict predicted-probability calibration.</p>
  </div>
</div>

<div id="tab-calculator" class="tab-panel">
  <div class="panel">
    <h2>Live Financial Calculator</h2>
    <p class="chart-story">Every slider below drives a live recomputation using the REAL true-positive
    (__TP_FLAGGED__) and false-positive (__FP_FLAGGED__) counts from Notebook 60's confusion matrix at the winning
    candidate, the real measured alert-month rate (__ALERT_MONTH_RATE__), and the real EAD/LGD inherited from
    Problem 1's Notebook 08 -- only the ASSUMPTION inputs below are adjustable.</p>
    <div class="calc-grid">
      <div>
        <div class="calc-slider-row">
          <label>Cohort-intervention success rate: <span class="val" id="successRateVal"></span></label>
          <input type="range" id="successRateSlider" min="0" max="60" step="1" style="width:100%;">
        </div>
        <div class="calc-slider-row">
          <label>Cost per cohort-review account (USD): <span class="val" id="reviewCostVal"></span></label>
          <input type="range" id="reviewCostSlider" min="0" max="100" step="1" style="width:100%;">
        </div>
        <div class="calc-slider-row">
          <label>Portfolio risk review cost per alert month (USD): <span class="val" id="portfolioCostVal"></span></label>
          <input type="range" id="portfolioCostSlider" min="0" max="10000" step="100" style="width:100%;">
        </div>
        <div class="calc-slider-row">
          <label>Annual application cycles: <span class="val" id="cyclesVal"></span></label>
          <input type="range" id="cyclesSlider" min="1" max="52" step="1" style="width:100%;">
        </div>
        <div class="calc-slider-row">
          <label>Implementation cost (USD): <span class="val" id="implCostVal"></span></label>
          <input type="range" id="implCostSlider" min="5000" max="150000" step="1000" style="width:100%;">
        </div>
      </div>
      <div class="calc-out">
        <div class="row"><span>Preventable defaults</span><span id="outPreventable"></span></div>
        <div class="row"><span>Gross loss prevented / cycle</span><span id="outGross"></span></div>
        <div class="row"><span>Cohort review cost / cycle</span><span id="outReview"></span></div>
        <div class="row"><span>Portfolio risk review cost / cycle (expected)</span><span id="outPortfolio"></span></div>
        <div class="row total"><span>Net benefit / cycle</span><span id="outNet"></span></div>
        <div class="row"><span>Annual net benefit</span><span id="outAnnual"></span></div>
        <div class="row total"><span>Year-1 ROI</span><span id="outRoi"></span></div>
        <div class="row total"><span>Payback period</span><span id="outPayback"></span></div>
      </div>
    </div>
  </div>
</div>

<div id="tab-smart" class="tab-panel">
  <div class="panel">
    <label for="orgFilter"><b>SMART Suggestions -- filter by organizational level (slicer)</b></label><br/>
    <select id="orgFilter"></select>
    <table id="smartTable"><thead><tr><th>Org Level</th><th>Suggestion</th></tr></thead><tbody></tbody></table>
  </div>
</div>

<script>
const candidateData = __CANDIDATE_JSON__;
const smartData = __SMART_JSON__;
const orgLevels = __ORG_LEVELS__;
const policyKv = __POLICY_KV_JSON__;
const validationKv = __VALIDATION_KV_JSON__;
const calc = __CALC_JSON__;
let feedData = __FEED_JSON__;

// --- Tab navigation ---
document.querySelectorAll(".tab-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    document.querySelectorAll(".tab-btn").forEach(b => b.classList.remove("active"));
    document.querySelectorAll(".tab-panel").forEach(p => p.classList.remove("active"));
    btn.classList.add("active");
    document.getElementById("tab-" + btn.dataset.tab).classList.add("active");
  });
});

// --- Policy / Validation key-value tables ---
function renderKvTable(tbodyEl, rows) {
  tbodyEl.innerHTML = "";
  rows.forEach(([k, v]) => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td><b>${k}</b></td><td>${v}</td>`;
    tbodyEl.appendChild(tr);
  });
}
renderKvTable(document.querySelector("#policyTable tbody"), policyKv);
renderKvTable(document.querySelector("#validationTable tbody"), validationKv);

// --- Candidate sweep: interactive chart + slicer + metric filter ---
const METRICS = [
  {key: "lift", label: "Default-Rate Lift", suffix: "x"},
  {key: "precision", label: "Precision", suffix: ""},
  {key: "recall", label: "Recall", suffix: ""},
  {key: "f1", label: "F1", suffix: ""},
  {key: "mcc", label: "MCC", suffix: ""},
];
let activeMetric = "lift";
let kpiOnly = false;

const metricButtonsEl = document.getElementById("metricButtons");
METRICS.forEach(m => {
  const b = document.createElement("button");
  b.className = "metric-btn" + (m.key === activeMetric ? " active" : "");
  b.textContent = m.label;
  b.dataset.metric = m.key;
  b.addEventListener("click", () => { activeMetric = m.key; refreshCandidateView(); });
  metricButtonsEl.appendChild(b);
});
document.getElementById("kpiOnlyFilter").addEventListener("change", (e) => {
  kpiOnly = e.target.checked; refreshCandidateView();
});

// Chart.js loads from a CDN -- if the viewer's network blocks it, the REST of this dashboard must still
// work. Every Chart.js call below is guarded so a missing library degrades gracefully.
let candChart = null;
if (typeof Chart !== "undefined") {
  try {
    const candCtx = document.getElementById("candidateChart").getContext("2d");
    candChart = new Chart(candCtx, {
      type: "bar",
      data: { labels: [], datasets: [{ label: "", data: [], backgroundColor: [] }] },
      options: {
        responsive: true,
        plugins: {
          legend: { display: true, position: "top" },
          tooltip: { callbacks: { label: (ctx) => `${ctx.dataset.label}: ${ctx.formattedValue}` } },
        },
        scales: { y: { beginAtZero: true } },
      },
    });
  } catch (e) { candChart = null; }
}
if (!candChart) {
  const chartEl = document.getElementById("candidateChart");
  if (chartEl) {
    chartEl.style.display = "none";
    const notice = document.createElement("p");
    notice.className = "chart-story";
    notice.textContent = "Chart.js could not load from the CDN in this environment (offline or blocked) -- "
      + "the interactive chart is unavailable, but the table below still reflects every filter and metric selection.";
    chartEl.after(notice);
  }
}

function refreshCandidateView() {
  document.querySelectorAll(".metric-btn[data-metric]").forEach(b => b.classList.toggle("active", b.dataset.metric === activeMetric));
  const metricMeta = METRICS.find(m => m.key === activeMetric);
  const visible = candidateData.filter(r => !kpiOnly || r.meets_kpi);
  if (candChart) {
    candChart.data.labels = visible.map(r => "Candidate " + r.candidate);
    candChart.data.datasets[0] = {
      label: metricMeta.label,
      data: visible.map(r => r[activeMetric]),
      backgroundColor: visible.map(r => r.is_winner ? "#C9A227" : (r.meets_kpi ? "#16a34a" : "#8A93A6")),
    };
    candChart.update();
  }

  document.getElementById("candidateLegend").innerHTML =
    `<span><span class="legend-swatch" style="background:#C9A227;"></span>Winning candidate</span>` +
    `<span><span class="legend-swatch" style="background:#16a34a;"></span>Meets KPI</span>` +
    `<span><span class="legend-swatch" style="background:#8A93A6;"></span>Does not meet KPI</span>`;

  const tbody = document.querySelector("#candidateTable tbody");
  tbody.innerHTML = "";
  visible.forEach(r => {
    const tr = document.createElement("tr");
    if (r.is_winner) tr.classList.add("winner-row");
    tr.innerHTML = `<td>${r.candidate}</td><td>${r.n_alerted.toLocaleString()}</td>` +
      `<td>${r.pct_alerted}%</td><td>${r.lift}x</td><td>${r.meets_kpi ? "Yes" : "No"}</td>` +
      `<td>${r.precision}</td><td>${r.recall}</td><td>${r.f1}</td><td>${r.mcc}</td>`;
    tbody.appendChild(tr);
  });
}
refreshCandidateView();

// --- Alert Feed tab: real ops-dashboard table, slicer, and optional live fetch ---
let alertOnly = false;
function renderFeed() {
  const tbody = document.querySelector("#feedTable tbody");
  tbody.innerHTML = "";
  feedData.filter(r => !alertOnly || r.alert).forEach(r => {
    const tr = document.createElement("tr");
    if (r.alert) tr.classList.add("alert-row");
    tr.innerHTML = `<td>${r.month}</td><td>${r.n_statements.toLocaleString()}</td>` +
      `<td>${r.n_unique_customers.toLocaleString()}</td><td>${r.baseline_eligible ? "Yes" : "No"}</td>` +
      `<td>${r.breach_count}</td><td>${r.run_length}</td><td>${r.alert ? "ALERT" : "--"}</td>`;
    tbody.appendChild(tr);
  });
}
document.getElementById("alertOnlyFilter").addEventListener("change", (e) => {
  alertOnly = e.target.checked; renderFeed();
});
renderFeed();

document.getElementById("liveFetchBtn").addEventListener("click", async () => {
  const base = document.getElementById("liveApiBase").value.trim().replace(/\/$/, "");
  const key = document.getElementById("liveApiKey").value.trim();
  const statusEl = document.getElementById("liveStatus");
  if (!base) { statusEl.textContent = "Enter a live deployment host first (e.g. https://host:8011)."; return; }
  statusEl.textContent = "Fetching live /alert-feed...";
  try {
    const resp = await fetch(base + "/alert-feed", { headers: key ? { "X-API-Key": key } : {} });
    if (!resp.ok) throw new Error("HTTP " + resp.status);
    const json = await resp.json();
    feedData = (json.months || []).map(m => ({
      month: m.month, n_statements: m.n_statements, n_unique_customers: m.n_unique_customers,
      baseline_eligible: m.baseline_eligible, breach_count: m.breach_count,
      run_length: m.consecutive_breach_run_length, alert: m.alert,
    }));
    renderFeed();
    statusEl.textContent = `Live: showing ${feedData.length} months from ${base}.`;
  } catch (e) {
    statusEl.textContent = "Could not reach a live deployment at that host (" + e.message + ") -- "
      + "still showing the embedded real-run snapshot below.";
  }
});

// --- SMART Suggestions filter ---
function renderSmart(filterLevel) {
  const tbody = document.querySelector("#smartTable tbody");
  tbody.innerHTML = "";
  smartData.filter(r => filterLevel === "All" || r.org_level === filterLevel).forEach(r => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${r.org_level}</td><td>${r.suggestion}</td>`;
    tbody.appendChild(tr);
  });
}
const orgSelect = document.getElementById("orgFilter");
["All", ...orgLevels].forEach(level => {
  const opt = document.createElement("option");
  opt.value = level; opt.textContent = level;
  orgSelect.appendChild(opt);
});
orgSelect.onchange = () => renderSmart(orgSelect.value);
renderSmart("All");

// --- Live financial calculator ---
const fmtUsd = (v) => "$" + Math.round(v).toLocaleString();
function updateCalculator() {
  const successRate = Number(document.getElementById("successRateSlider").value) / 100;
  const reviewCost = Number(document.getElementById("reviewCostSlider").value);
  const portfolioCost = Number(document.getElementById("portfolioCostSlider").value);
  const cycles = Number(document.getElementById("cyclesSlider").value);
  const implCost = Number(document.getElementById("implCostSlider").value);

  document.getElementById("successRateVal").textContent = (successRate * 100).toFixed(0) + "%";
  document.getElementById("reviewCostVal").textContent = "$" + reviewCost;
  document.getElementById("portfolioCostVal").textContent = fmtUsd(portfolioCost);
  document.getElementById("cyclesVal").textContent = cycles + "x / year";
  document.getElementById("implCostVal").textContent = fmtUsd(implCost);

  const preventable = Math.round(calc.tp * successRate);
  const gross = preventable * calc.ead * calc.lgd;
  const reviewTotal = calc.fp * reviewCost;
  const portfolioTotal = calc.alert_month_rate * portfolioCost;
  const net = gross - reviewTotal - portfolioTotal;
  const annual = net * cycles;
  const roi = implCost > 0 ? ((annual - implCost) / implCost) * 100 : null;
  const payback = annual > 0 ? (implCost / (annual / 12)) : null;

  document.getElementById("outPreventable").textContent = preventable.toLocaleString();
  document.getElementById("outGross").textContent = fmtUsd(gross);
  document.getElementById("outReview").textContent = fmtUsd(reviewTotal);
  document.getElementById("outPortfolio").textContent = fmtUsd(portfolioTotal);
  document.getElementById("outNet").textContent = fmtUsd(net);
  document.getElementById("outAnnual").textContent = fmtUsd(annual);
  document.getElementById("outRoi").textContent = roi !== null ? roi.toFixed(0) + "%" : "N/A";
  document.getElementById("outPayback").textContent = payback !== null ? payback.toFixed(1) + " months" : "N/A -- no measurable net benefit";
}
["successRateSlider", "reviewCostSlider", "portfolioCostSlider", "cyclesSlider", "implCostSlider"].forEach(id => {
  document.getElementById(id).addEventListener("input", updateCalculator);
});
document.getElementById("successRateSlider").value = Math.round(calc.default_success_rate * 100);
document.getElementById("reviewCostSlider").value = calc.default_review_cost;
document.getElementById("portfolioCostSlider").value = calc.default_portfolio_review_cost;
document.getElementById("cyclesSlider").value = calc.default_cycles;
document.getElementById("implCostSlider").value = calc.default_impl_cost;
updateCalculator();
</script>
</body>
</html>
"""

_html = (_html
         .replace("__WINNING_CANDIDATE__", str(WINNING_CONSECUTIVE_BREACH_CANDIDATE))
         .replace("__N_CANDIDATES__", str(len(CANDIDATE_RESULTS)))
         .replace("__N_ALERT_MONTHS__", str(N_ALERT_MONTHS_TOTAL))
         .replace("__N_ELIGIBLE_MONTHS__", str(N_BASELINE_ELIGIBLE_MONTHS))
         .replace("__ALERT_MONTH_RATE__", f"{ALERT_MONTH_RATE:.1%}")
         .replace("__TP_FLAGGED__", f"{TRUE_POSITIVES_FLAGGED:,}")
         .replace("__FP_FLAGGED__", f"{FALSE_POSITIVES_FLAGGED:,}")
         .replace("__LIFT_POINT__", f"{(WINNING_METRICS['default_rate_lift'] or 0.0):.2f}x")
         .replace("__LIFT_CI__", f"[{LIFT_95CI[0]}, {LIFT_95CI[1]}]")
         .replace("__NET_BENEFIT__", f"${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
         .replace("__ROI__", ROI_DISPLAY)
         .replace("__PAYBACK__", PAYBACK_DISPLAY)
         .replace("__STATUS__", "RECOMMENDED" if RECOMMENDED_FOR_PRODUCTION else "NOT RECOMMENDED")
         .replace("__STATUS_COLOR__", "#16a34a" if RECOMMENDED_FOR_PRODUCTION else "#dc2626")
         .replace("__FINANCIAL_B64__", _financial_b64)
         .replace("__TREND_B64__", _trend_b64)
         .replace("__ROC_B64__", _roc_b64)
         .replace("__PR_B64__", _pr_b64)
         .replace("__LIFT_B64__", _lift_b64)
         .replace("__BOOTSTRAP_B64__", _bootstrap_b64)
         .replace("__CALIBRATION_B64__", _calibration_b64)
         .replace("__CANDIDATE_JSON__", json.dumps(_candidate_records))
         .replace("__SMART_JSON__", json.dumps(SMART_SUGGESTIONS))
         .replace("__ORG_LEVELS__", json.dumps(_org_levels))
         .replace("__POLICY_KV_JSON__", json.dumps(_policy_kv))
         .replace("__VALIDATION_KV_JSON__", json.dumps(_validation_kv))
         .replace("__CALC_JSON__", json.dumps(_calc_constants))
         .replace("__FEED_JSON__", json.dumps(_feed_records)))

dashboard_path = P11_REPORTING_DIR / "real_time_portfolio_monitoring_ops_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"\u2705 Saved -> {dashboard_path.name} ({dashboard_path.stat().st_size / 1e3:.1f} KB)")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: VERIFICATION
# =============================================================================
_section("SECTION 13: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("True positives + false negatives equals the real holdout defaulter count",
       TRUE_POSITIVES_FLAGGED + _cm_winning["fn"] == N_HOLDOUT_DEFAULTERS)
_check("Preventable defaults does not exceed true positives flagged",
       PREVENTABLE_DEFAULTS <= TRUE_POSITIVES_FLAGGED)
_check("Net benefit per cycle equals gross loss prevented minus cohort-review minus portfolio-risk-review cost",
       abs(NET_BENEFIT_PER_CYCLE_USD
           - (GROSS_LOSS_PREVENTED_USD - COHORT_REVIEW_COST_USD - PORTFOLIO_RISK_REVIEW_COST_PER_CYCLE_USD)) < 1e-6)
_check("Payback is a positive finite number when there is measurable annual net benefit, "
       "and explicitly undefined (None) otherwise -- never a crash or a fabricated value",
       (PAYBACK_MONTHS is not None and PAYBACK_MONTHS > 0) if ANNUAL_BENEFIT_USD > 0
       else PAYBACK_MONTHS is None)
_check("EAD/LGD were inherited from Notebook 08, not re-guessed",
       EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
       and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])
_check("Confusion-matrix counts were reused verbatim from Notebook 60 (not re-derived)",
       TRUE_POSITIVES_FLAGGED == NB60_SUMMARY["winning_candidate_metrics"]["confusion_matrix"]["tp"])
_check("Candidate sweep table covers every real candidate from Notebook 58's policy",
       set(CANDIDATE_RESULTS.keys()) == set(int(c) for c in PORTFOLIO_MONITORING_POLICY["consecutive_breach_candidates"]))
_check("Alert-month rate is a valid probability in [0, 1]", 0.0 <= ALERT_MONTH_RATE <= 1.0)
_check("Real alert-feed month count matches this notebook's own third reproduction of the calendar-month store",
       ALERT_FEED["n_months"] == _n_months)
_check("HTML dashboard embeds all 6 reused/new charts as self-contained base64 data URIs (portable, no "
       "broken relative paths)",
       all(b for b in [_trend_b64, _roc_b64, _pr_b64, _lift_b64, _bootstrap_b64, _calibration_b64, _financial_b64]))

_expected_files = [assumptions_path, smart_path, chart_financial_path, report_path, workbook_path, dashboard_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 61 verification checks failed. See \u274c line above.")
print("\nAll Notebook 61 checks passed.")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 61 SUMMARY -- PROBLEM 11 COMPLETE
# =============================================================================
_section("SECTION 14: Write Notebook 61 Summary -- Problem 11 Complete")

notebook_61_summary = {
    "notebook": "61_real_time_portfolio_monitoring_financial_impact_reporting_packaging",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 11, "problem_name": "Real-Time Portfolio Monitoring",
    "phase": "Phase 4 -- Operational Risk Management", "problem_11_complete": True,
    "winning_consecutive_breach_candidate": WINNING_CONSECUTIVE_BREACH_CANDIDATE, "meets_kpi_target": MEETS_KPI,
    "recommended_for_production": RECOMMENDED_FOR_PRODUCTION,
    "true_positives_flagged": TRUE_POSITIVES_FLAGGED, "false_positives_flagged": FALSE_POSITIVES_FLAGGED,
    "real_cohort_capture_rate": round(COHORT_CAPTURE_RATE, 4),
    "real_alert_months": N_ALERT_MONTHS_TOTAL, "real_baseline_eligible_months": N_BASELINE_ELIGIBLE_MONTHS,
    "real_alert_month_rate": round(ALERT_MONTH_RATE, 4),
    "net_benefit_per_cycle_usd": round(NET_BENEFIT_PER_CYCLE_USD, 2),
    "roi_year_1_pct": ROI_PCT_JSON, "payback_period_months": PAYBACK_MONTHS_JSON,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb61_summary_path = ARTIFACTS_DIR / "notebook_61_summary.json"
with open(nb61_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_61_summary, f, indent=2)
print(f"\u2705 Saved -> {nb61_summary_path.name}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: COMPLETION SUMMARY -- PROBLEM 11 & PHASE 4 COMPLETE
# =============================================================================
_section("SECTION 15: Notebook 61 Complete -- Problem 11 and Phase 4 Complete")

print("NOTEBOOK 61: FINANCIAL-IMPACT REPORTING & PACKAGING (ELEVATED) -- COMPLETE")
print("PROBLEM 11 (REAL-TIME PORTFOLIO MONITORING) -- ALL 4 NOTEBOOKS COMPLETE")
print("PHASE 4 (OPERATIONAL RISK MANAGEMENT) -- ALL 3 PROBLEMS COMPLETE (9, 10, 11)")
print(f"  Winning candidate / meets KPI / recommended : {WINNING_CONSECUTIVE_BREACH_CANDIDATE} / {MEETS_KPI} / "
      f"{RECOMMENDED_FOR_PRODUCTION}")
print(f"  Real defaulters captured (exact)             : {TRUE_POSITIVES_FLAGGED:,} of "
      f"{N_HOLDOUT_DEFAULTERS:,} ({COHORT_CAPTURE_RATE:.1%})")
print(f"  Real cohort default-rate lift (95% CI)        : {(WINNING_METRICS['default_rate_lift'] or 0.0):.2f}x "
      f"[{LIFT_95CI[0]}, {LIFT_95CI[1]}]")
print(f"  Real alert months (of baseline-eligible)      : {N_ALERT_MONTHS_TOTAL} of {N_BASELINE_ELIGIBLE_MONTHS} "
      f"({ALERT_MONTH_RATE:.1%})")
print(f"  Net benefit per cycle                         : ${NET_BENEFIT_PER_CYCLE_USD:,.0f}")
print(f"  Estimated Year-1 ROI / payback                 : {ROI_DISPLAY} / {PAYBACK_DISPLAY}")
print(f"  Word report (elevated, synthesizes NB58-60)    : {report_path.name}")
print(f"  Excel workbook (colorful, tables, charts)      : {workbook_path.name}")
print(f"  HTML dashboard (elevated, doubles as real ops dashboard + alert feed): {dashboard_path.name}")
print(f"  Files produced                                : {len(_expected_files) + 1}")
for _p in _expected_files + [nb61_summary_path]:
    print(f"    - {_p.name}")
print("\n  PROBLEM 11 (Real-Time Portfolio Monitoring) is now complete, closing out PHASE 4 (Operational Risk "
      "Management) -- Problems 9 (Collections Optimization), 10 (Credit Line Management), and 11 (Real-Time "
      "Portfolio Monitoring) are all now complete.")
print("\n\u2705 Ready to proceed to the next phase of the master plan.")
